# CFSL vs QFSL (+ TTDA) — Track A (classical, rigorously corrected · round 3)

One notebook, split by section. Runs off the Stage-3 causal/spur banks + frozen CheXzero; **no `best_model.pt` / DDE needed**. Quantum stays a stubbed `φ_q` at the injection point in §2 — every artifact it needs (identical **in-domain AND cross-domain** episodes incl. anchor tensors, CheXpert cache, classical baselines) is saved in §8/§9.

**Round-2 corrections are applied and annotated inline** (search `[R n]`), on top of the original 28-point audit (`[n]`):

- **`[R1]` TTDA ladders are truly one-change-per-row** — the φ rows now KEEP the transduction the previous row added (`eval_phi(transductive=True)`), so φ is the only new factor in both the in-domain and cross-domain ladders. Cross-domain φ rows also sit on the same spur-cleaned+anchored base.
- **`[R2]` `adapt_phi` pseudo-query rng is threaded per-episode** — a unique `default_rng(seed·1e6+episode)` per episode; the held-out pseudo-query genuinely varies. `adapt_phi` now *raises* if no rng is passed.
- **`[R3]` TENT uses a sharp `TENT_TEMP=0.1`** (not `COS_TEMP=1.0`) so the entropy gradient is non-vanishing and adaptation can move φ.
- **`[R4]` Cross-domain anchoring blends NIH + support prototypes in the raw 512-d space and projects once through φ** — NIH vectors are never pushed through a CheXpert-fit projection into an unrelated coordinate system.
- **`[R5]`/`[R21]` FULLBANK-enforced, coverage-matched ablation** — causal-only vs **spur-only (no causal)** vs all-patch concat vs causal_full, all on images present in every bank (equal N), removing the content-vs-coverage confound.
- **`[R6]` Paired Wilcoxon signed-rank** between every adjacent/reference row (per-episode AUROC) with `* / ** / ***` stars.
- **`[R7]` Bootstrap percentile 95% CI** replaces the normal approximation.
- **`[R8]` Seed image-overlap fraction** measured and logged. **`[R9]` Per-class AUROC** alongside macro. **`[R10]` Linear-probe (LR)** baseline.
- **`[R11]` α-anchor validation sweep** and **`[R12]` φ weight-decay sweep** on held-out validation seeds; chosen values locked and reported. **`[R13]` "no meta-training"** and **`[R14]` multi-label contamination** are quantified, not hidden.
- **`[R15]`–`[R20]` Infrastructure**: reproducibility block, per-seed checkpoint/resume, wall-clock timing, pre-flight sanity asserts, pinned CLIP commit, and cross-domain episode dump.

### Round-3 corrections (search `[#n]`) — the silent-but-real bugs

- **`[#1]` Patient-disjointness is now episode-scoped, not per-class.** A two-pass build in `gen_episodes` collects support patients across **all** classes first, then admits queries only for patients in no class's support — closing the cross-class channel where patient P could be in class-A support and class-B query. `_sanity_check` now asserts patient-level (not just image-level) disjointness.
- **`[#2]` φ weight-init is seeded per episode.** `build_phi(init_seed=…)` reinitialises each `nn.Linear` from a dedicated `torch.Generator` (same `seed·1e6+episode` used for `adapt_phi`), so φ-adapted numbers no longer depend on execution order and `[R16]` resume reproduces an uninterrupted run. `_sanity_check` asserts identical weights for a fixed seed.
- **`[#3]` HP validation holds out IMAGES, not just a seed.** `VAL_FRAC` of each class is reserved (patient-aware) **before** balancing and excluded from `bal`/`cx_bal`; `PHI_WD` and `CX_ANCHOR_ALPHA` are tuned only on that disjoint pool.
- **`[#4]` Holm–Bonferroni** correction across each family of paired Wilcoxon tests (raw and corrected stars both shown, family size reported).
- **`[#5]` Cluster (block) bootstrap** — resample whole seeds then episodes — is the reported CI, since `[R8]` shows episodes are not i.i.d.
- **`[#6]` Class-split robustness**: Ours-full reported as mean ± std across several `TA_SPLIT_SEED` partitions, not one.
- **`[#7]`** dead `CX_TEST_FRAC` removed. **`[#8]`** the two distinct meanings of "clean" (per-image in-domain vs dataset-mean cross-domain) documented at both call sites. **`[#9]`** the frozen-φ row relabelled **"frozen random-init"** (it is an untrained random projection, a sanity floor). **`[#10]`** CheXpert load failures and patient-regex fallbacks are counted and logged. Chosen `PHI_WD`/`α` are **threaded explicitly** (no global mutation); checkpoint tags are collision-checked; TENT's train-on-query-batch behaviour is disclosed in-code.

> Set **`DRY_RUN = True`** for a smoke test first, then `False` for the full protocol (200 episodes × 5 seeds × K∈{1..5}).
>
> **Honest framing kept:** the zero-param cosine prototype is the robust reference; φ-learners are expected to help mainly as K grows (the sample-efficiency gap a quantum `φ_q` must beat). In-domain "TTDA" has no real domain gap and stays a flat control. With 5 classes (3 test-novel) there are ~0 base-class episodes, so φ is **support-adapted, not meta-trained** — CFSL numbers are a lower bound, quantified in `[R13]`. Remaining unswept magic numbers (`TA_BETA`, `ADAPT_TEMP`, `TENT_LR`) are flagged as such rather than presented as tuned.

## Setup & Config
Dependencies, device (CUDA→CPU fallback), paths, Track-A protocol constants, and the φ injection-point knobs. **Set `DRY_RUN=False` for the full run** and verify the bank/CheXpert paths.

In [ ]:
# ==============================================================================
# CFSL vs QFSL (+ TTDA)  —  Few-Shot Test-Time Adaptation on Causal Patch Banks
# In-domain NIH (Track-A protocol)  +  Cross-domain NIH -> CheXpert
# ------------------------------------------------------------------------------
# CLASSICAL, RIGOROUSLY CORRECTED BUILD.  φ_q (quantum) stays a clean stub at the
# same injection point; every artifact the quantum cell needs is saved to disk.
#
# Runs entirely off Stage-3 patch banks (CheXzero-512 space) + frozen CheXzero for
# CheXpert. Does NOT need best_model.pt / the trained DDE. Saves every .pt/.json/.csv
# (including the *exact episodes* as precomputed feature tensors, in-domain AND
# cross-domain) so the quantum notebook can score φ_q on identical episodes offline.
#
# ------------------------------------------------------------------------------
# ROUND-2 FIXES (on top of the 28-point audit).  Search `[R#]` for the fix.
#   [R1]  TTDA ladders are now TRULY one-change-per-row: the φ rows KEEP the
#         transduction the previous row added (eval_phi(transductive=True)); φ is the
#         only new factor, in both the in-domain and cross-domain ladders.
#   [R2]  adapt_phi pseudo-query rng is threaded per-episode (unique rng per
#         (seed, episode)); the held-out pseudo-query truly varies across episodes.
#   [R3]  TENT uses a sharp TENT_TEMP (=0.1), NOT COS_TEMP=1.0, so the entropy
#         gradient is non-vanishing and adaptation can actually move φ.
#   [R4]  eval_phi anchoring blends NIH + support prototypes in the RAW 512-d space
#         and projects ONCE through φ (no NIH vectors pushed through a CheXpert-fit
#         projection into an unrelated coordinate system).
#   [R5]  FULLBANK is ENFORCED: a `bal_full` pool restricted to imgs present in all
#         three banks drives a dedicated coverage-controlled ablation (equal N).
#   [R6]  Paired Wilcoxon signed-rank test between every adjacent ladder/table row
#         (per-episode AUROC), reported with * / ** / *** significance stars.
#   [R7]  Bootstrap percentile 95% CI (BOOTSTRAP_N resamples) replaces the normal
#         approximation for the pooled AUROC interval.
#   [R8]  Episode image-overlap fraction across seeds is measured and logged to JSON.
#   [R9]  Per-class AUROC reported alongside the macro average.
#   [R10] Linear-probe (logistic-regression on frozen support features) baseline.
#   [R11] CX_ANCHOR_ALPHA validation sweep on held-out val episodes; α is locked to
#         the best value before the main cross-domain run (reported in an appendix).
#   [R12] PHI_WD sensitivity sweep on val-novel episodes; chosen WD reported.
#   [R13] "No meta-training" is quantified: base/val/test class counts + theoretical
#         base-episode budget (0) are computed and logged as a measured limitation.
#   [R14] Multi-label contamination rate of the mined single-label pool is measured.
#   [R15] Full reproducibility block (library + CUDA + GPU + RNG state) dumped to JSON.
#   [R16] Per-seed checkpointing: each method/seed result is written to CKPT_DIR and
#         reloaded on restart so an OOM/timeout does not lose completed work.
#   [R17] Wall-clock seconds recorded per method (classical timing baseline for QFSL).
#   [R18] _sanity_check() runs a 2-episode smoke test with hard asserts (finite AUROC
#         in [0,1], disjoint support/query keys, ways length, anchor shape) first.
#   [R19] Dependency pinning: CLIP pinned to a commit hash; sklearn/scipy asserted.
#   [R20] Cross-domain episodes (incl. per-episode anchor tensors) are dumped too.
#   [R21] Extra ablation: causal-only vs SPUR-ONLY (spin⊕spout, no causal) vs
#         all-patch concat vs causal_full, all on the coverage-matched FULLBANK pool.
#
# HONEST FRAMING:
#   * Zero-param cosine prototype is the robust reference. φ-learners are expected to
#     help as K grows and may lose at K=1 — the sample-efficiency gap a quantum φ_q
#     must beat. A truly-frozen φ_c row stays in: if adapted≈frozen, learning adds nothing.
#   * In-domain "TTDA" has no real domain gap -> ~flat; kept as a control.
#   * CheXpert is natively multi-label; the 3-way single-label restriction is a stated
#     protocol choice for comparability, footnoted not hidden.
#   * With 5 target classes (3 test-novel) there are ~0 base-class episodes: φ is
#     support-adapted, NOT meta-trained. This is quantified in [R13], not hidden.
#
# ⚠️ DRY_RUN=True first (fast smoke test). Flip to False for the full protocol.
# ==============================================================================

import os, gc, re, sys, json, math, glob, time, copy, platform, warnings, csv as _csv
from collections import defaultdict as _dd
warnings.filterwarnings("ignore")

# ------------------------------------------------------------------ dependencies
import subprocess
def _pip(*a): subprocess.run([sys.executable, "-m", "pip", "install", "-q", *a], check=False)

# [R19] Pin CLIP to a fixed commit so a silent upstream change cannot alter features.
CLIP_COMMIT = "a9b1bf5920416aaeaec965c25dd9e8f98c864f16"  # openai/CLIP, pinned
try:
    import clip  # openai CLIP for CheXzero
except Exception:
    _pip(f"git+https://github.com/openai/CLIP.git@{CLIP_COMMIT}", "ftfy", "regex")
    import clip

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from tqdm import tqdm
import sklearn
from sklearn.metrics import roc_auc_score, f1_score
from sklearn.linear_model import LogisticRegression
import scipy
from scipy.stats import wilcoxon

# [R19] sklearn/scipy edge-case behaviour (roc_auc_score, wilcoxon) is version
# sensitive; assert a floor rather than silently scoring on an unexpected version.
assert tuple(int(x) for x in sklearn.__version__.split(".")[:2]) >= (1, 0), \
    f"scikit-learn >= 1.0 required, found {sklearn.__version__}"
assert tuple(int(x) for x in scipy.__version__.split(".")[:2]) >= (1, 5), \
    f"scipy >= 1.5 required, found {scipy.__version__}"

# CUDA preferred (Kaggle GPU); CPU fallback so the same file reloads/scores offline.
if torch.cuda.is_available():
    Device = torch.device("cuda"); torch.backends.cudnn.benchmark = True
else:
    Device = torch.device("cpu")
    print("⚠️  CUDA not found — running on CPU (fine for reload/scoring; slow for full run).")

In [ ]:
# ==============================================================================
# SECTION 0 — CONFIG
# ==============================================================================
DRY_RUN = True            # ← smoke test (few episodes/seeds). Set False for full run.
RUN_INDOMAIN    = True
RUN_CROSSDOMAIN = True
RUN_QUANTUM     = False   # QFSL φ_q stub — leave False until the quantum cell lands.
SAVE_EPISODES   = True    # dump exact episodes (feature tensors) for the quantum notebook.

# ---- paths (verify before the full run) ----
PATCH_DIR = "/kaggle/input/datasets/anikazarin/nih-patch-output-20thjuly/output"
STAGE3_CAUSAL   = f"{PATCH_DIR}/stage3_causal.pt"
STAGE3_SPUR_IN  = f"{PATCH_DIR}/stage3_spur_in.pt"
STAGE3_SPUR_OUT = f"{PATCH_DIR}/stage3_spur_out.pt"

CHEXPERT_DIR  = "/kaggle/input/datasets/ashery/chexpert"
CHEXZERO_CKPT = ("/kaggle/input/models/anikazarin/chexzero/pytorch/default/1/"
                 "best_64_5e-05_original_22000_0.864.pt")
# Optional: full NIH image-level labels for single-label verification (fix #5).
# Set to a Data_Entry_2017.csv path if available; None => skip cross-check (logged).
NIH_LABEL_CSV = None

OUT_DIR = "/kaggle/working"; os.makedirs(OUT_DIR, exist_ok=True)
CX_CACHE = f"{OUT_DIR}/chexpert_frontal_cz512.pt"
EPISODE_DIR = f"{OUT_DIR}/episodes"; os.makedirs(EPISODE_DIR, exist_ok=True)
CKPT_DIR = f"{OUT_DIR}/checkpoints"; os.makedirs(CKPT_DIR, exist_ok=True)  # [R16]

TARGET_CLASSES = ["Atelectasis", "Cardiomegaly", "Consolidation", "Edema", "Pleural Effusion"]
N_CLASSES = len(TARGET_CLASSES)
CX_ALIASES = {"Atelectasis": ["Atelectasis"], "Cardiomegaly": ["Cardiomegaly"],
              "Consolidation": ["Consolidation"], "Edema": ["Edema"],
              "Pleural Effusion": ["Pleural Effusion", "Effusion", "Pleural_Effusion"]}

# ---- Track-A protocol ----
TA_WAY, TA_SHOT, TA_QUERY = 3, 5, 15
TA_MIN_QUERY = TA_QUERY       # [16] episodes must supply the full query budget
TA_EPISODES = 200
TA_SEEDS    = [0, 1, 2, 3, 4]
TA_SPLIT_SEED = 123           # fixed base/val-novel/test-novel class split
TA_BETA = 0.5                 # spur subtraction strength
TA_QT_MARGIN = 0.05           # [12] transductive gate: min (top1-top2) cosine margin
K_SWEEP = [1, 2, 3, 4, 5]     # sample-efficiency axis
COS_TEMP = 1.0                # [10] softmax temp ONLY where probabilities are needed

# ---- statistics (round-2) ----
BOOTSTRAP_N = 10000           # [R7] bootstrap resamples for percentile CI
BOOT_SEED   = 20240717        # fixed rng for reproducible bootstrap/wilcoxon
WILCOXON_MIN_N = 10           # [R6] min paired episodes to run a signed-rank test
SANITY_EPISODES = 2           # [R18] pre-flight smoke episodes per method

# ---- φ (injection point) ----
N_QUBITS  = 8                 # 512 -> n_qubits reduce; 8-qubit VQC later
PHI_DEPTH = 2
PHI_STEPS_PER_SHOT = 15       # [9] adapt steps scale with K (steps = clip(K*this, ...))
PHI_STEPS_MIN, PHI_STEPS_MAX = 20, 90
PHI_LR    = 5e-3
PHI_WD    = 1e-3              # strong reg — support sets are tiny (validated in [R12])
PHI_WD_SWEEP = [1e-4, 1e-3, 1e-2]   # [R12] WD sensitivity sweep on val-novel episodes
TENT_STEPS = 20
TENT_LR    = 1e-3
TENT_TEMP  = 0.1              # [R3] SHARP temp inside TENT only (COS_TEMP stays 1.0 for AUROC)
ADAPT_TEMP = 0.5             # temp for the adaptation CE loss (training only)

# ---- cross-domain ----
CX_ANCHOR_ALPHA = 0.5         # DEFAULT blend α·NIH-proto + (1-α)·support-proto; the value
                              # actually used is chosen by the [R11] sweep and THREADED
                              # explicitly (no global mutation — round-3 #minor).
ALPHA_SWEEP = [0.1, 0.3, 0.5, 0.7, 0.9]   # [R11] validated on an IMAGE-held-out val split
CX_SUBSET_ENCODE = None       # None = all frontal; or an int to cap encoding

# ---- round-3: image-level held-out validation split (fixes HP leakage #3) ----
VAL_FRAC = 0.20               # per-class fraction reserved for HP tuning, EXCLUDED from
                              # the main-run pools (bal/cx_bal) so PHI_WD/α are tuned on
                              # images that never appear in a reported episode.
N_SPLIT_ROBUST = 5            # [#6] class-split seeds for the split-robustness report

SEED = 42
np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)

if DRY_RUN:
    TA_EPISODES, TA_SEEDS, K_SWEEP = 20, [0, 1], [1, 5]
    PHI_STEPS_MAX, TENT_STEPS = 40, 8
    TA_MIN_QUERY = 5             # relax so the tiny smoke test still forms episodes
    CX_SUBSET_ENCODE = 4000
    BOOTSTRAP_N = 2000
    ALPHA_SWEEP = [0.3, 0.5, 0.7]
    PHI_WD_SWEEP = [1e-3, 1e-2]
    print("⚠️  DRY_RUN — reduced episodes/seeds/K/min-query/bootstrap. "
          "Flip DRY_RUN=False for the real run.\n")

def _bn(s): return str(s).replace("\\", "/").split("/")[-1].strip()

def _phi_steps(K):
    return int(np.clip(K * PHI_STEPS_PER_SHOT, PHI_STEPS_MIN, PHI_STEPS_MAX))

def _nih_patient(key):
    """NIH filenames encode patient as the digits before the first underscore."""
    m = re.match(r"(\d+)_", _bn(key))
    return m.group(1) if m else _bn(key)

## §1 — Banks → signatures
Loads the three Stage-3 patch banks (CheXzero-512 space), builds frontal single-label pools, the fixed class split (seed 123), and balanced test-novel pools. No `best_model.pt` / NIH images needed.

In [ ]:
# ==============================================================================
# SECTION 1 — LOAD BANKS + BUILD SIGNATURES (frontal, single-label pools)
# ==============================================================================
print("=" * 70, "\n[1] Loading Stage-3 banks + building signatures\n", "=" * 70)

def load_bank(path, name):
    if not os.path.isfile(path):
        print(f"  [bank] {name:<9}: MISSING ({path})"); return torch.zeros(0, 512), []
    b = torch.load(path, map_location="cpu", weights_only=False)
    if isinstance(b, dict) and "embeddings" in b and "meta" in b:
        e, m = b["embeddings"].float(), b["meta"]
    elif isinstance(b, dict) and "causal_embeddings" in b:
        e, m = b["causal_embeddings"].float(), b["causal_meta"]
    else:
        raise ValueError(f"{name}: unknown bank schema {list(b)}")
    assert e.shape[0] == len(m), f"{name}: emb/meta length mismatch"
    print(f"  [bank] {name:<9}: {tuple(e.shape)} ({len(m):,} patches)")
    return e, m

def _frontal_ok(key):                 # NIH names carry no view token -> frontal
    return "lateral" not in key.lower()

def build_signatures(path, name):
    """image_key -> (L2-normalised sig, RAW mean-pooled sig); + mined class set."""
    e, m = load_bank(path, name)
    by, mined = _dd(list), _dd(set)
    for emb, mm in zip(e, m):
        k = _bn(mm.get("image_name", ""))
        if not k or not _frontal_ok(k): continue
        by[k].append(emb)
        p = mm.get("pathology")
        if p in TARGET_CLASSES: mined[k].add(p)
    sig, sig_raw = {}, {}
    for k, v in by.items():
        pooled = torch.stack(v).mean(0)
        sig_raw[k] = pooled                         # [13] unnormalised for Euclidean baseline
        sig[k] = F.normalize(pooled, dim=0)
    return sig, sig_raw, mined

sig_causal, sig_causal_raw, mined = build_signatures(STAGE3_CAUSAL, "causal")
sig_spin, _, _                    = build_signatures(STAGE3_SPUR_IN, "spur_in")
sig_spout, _, _                   = build_signatures(STAGE3_SPUR_OUT, "spur_out")
assert sig_causal, "❌ causal signatures empty — check STAGE3_CAUSAL path."
HAS_SPOUT = len(sig_spout) > 0

# [15] all-patch signature = mean(causal, spin, spout); track which imgs have all 3 banks.
# [R21] spur-only signature = mean(spin, spout) with NO causal channel — the "without
#       causal, spurious patches only" concatenated ablation the reviewer asked for.
sig_all, sig_spuronly = {}, {}
FULLBANK = set()                    # imgs present in causal ∧ spin ∧ spout
for k in sig_causal:
    parts = [sig_causal[k]]
    if k in sig_spin:  parts.append(sig_spin[k])
    if k in sig_spout: parts.append(sig_spout[k])
    if k in sig_spin and k in sig_spout:
        FULLBANK.add(k)
        sig_spuronly[k] = F.normalize(torch.stack([sig_spin[k], sig_spout[k]]).mean(0), dim=0)
    sig_all[k] = F.normalize(torch.stack(parts).mean(0), dim=0)
print(f"  imgs with all 3 banks (all-patch / spur-only / causal comparable here): "
      f"{len(FULLBANK):,}/{len(sig_causal):,}")

# global spur_out direction (generic hard-negative) for cross-domain cleaning
spout_dir = F.normalize(torch.stack(list(sig_spout.values())).mean(0), dim=0) if HAS_SPOUT else None

# ---- single-label pool (exactly one mined finding) ----
# [5] mined labels come from patch metadata. If NIH_LABEL_CSV is provided, cross-check
#     against image-level ground truth and drop images that are actually multi-label.
nih_img_labels = None
if NIH_LABEL_CSV and os.path.isfile(NIH_LABEL_CSV):
    _df = pd.read_csv(NIH_LABEL_CSV)
    _pcol = "Image Index" if "Image Index" in _df.columns else _df.columns[0]
    _lcol = "Finding Labels" if "Finding Labels" in _df.columns else _df.columns[1]
    nih_img_labels = {}
    for _, r in _df.iterrows():
        lbls = {x.strip() for x in str(r[_lcol]).split("|")}
        nih_img_labels[_bn(r[_pcol])] = {c for c in TARGET_CLASSES if c in lbls}
    print(f"  [5] NIH image-level labels loaded for cross-check: {len(nih_img_labels):,} imgs")
else:
    print("  [5] NIH_LABEL_CSV not set — single-label pool derived from patch metadata only "
          "(possible multi-label contamination; measured below).")

# [R14] Multi-label contamination rate of the mined pool. This directly bounds the
#       noise floor on episode label quality and is logged to the results JSON.
_n_mined = max(1, len(mined))
_n_multi = sum(1 for k, v in mined.items() if len(v) > 1)
_n_single_mined = sum(1 for k, v in mined.items() if len(v) == 1)
CONTAMINATION = {
    "n_mined_imgs": len(mined),
    "n_multilabel_patchmeta": _n_multi,
    "frac_multilabel_patchmeta": _n_multi / _n_mined,
    "n_single_patchmeta": _n_single_mined,
    "nih_label_csv_available": nih_img_labels is not None,
}
print(f"  [R14] contamination: {_n_multi}/{len(mined)} "
      f"({100*_n_multi/_n_mined:.1f}%) mined imgs had co-occurring findings in patch "
      f"metadata; {_n_single_mined} are patch-single. "
      f"{'Cross-checked vs image-level labels.' if nih_img_labels is not None else 'Unverified without NIH_LABEL_CSV.'}")

def _is_single_label(k, mined_set):
    if len(mined_set) != 1: return False
    if nih_img_labels is not None and k in nih_img_labels:
        return len(nih_img_labels[k]) == 1 and next(iter(mined_set)) in nih_img_labels[k]
    return True

single, _dropped_ml = {}, 0
for k, v in mined.items():
    if k not in sig_causal: continue
    if _is_single_label(k, v):
        single[k] = next(iter(v))
    elif len(v) == 1:
        _dropped_ml += 1
CONTAMINATION["n_dropped_multilabel_imglevel"] = _dropped_ml
pool = _dd(list)
for k, c in single.items(): pool[TARGET_CLASSES.index(c)].append(k)
if nih_img_labels is not None:
    print(f"  [5] dropped {_dropped_ml} patch-single imgs that are multi-label at image level")
print("  single-label pool per class: " +
      "  ".join(f"{TARGET_CLASSES[c][:4]}={len(pool[c])}" for c in range(N_CLASSES)))

# NIH patient map for in-domain de-duplication [8]
nih_patient_of = {k: _nih_patient(k) for k in single}
# [#10] count images whose patient regex did NOT match (fall back to filename == its own
#       singleton "patient", which silently weakens de-duplication for that subset).
_pat_fallback = sum(1 for k in single if not re.match(r"(\d+)_", _bn(k)))
CONTAMINATION["n_patient_regex_fallback"] = _pat_fallback
CONTAMINATION["frac_patient_regex_fallback"] = _pat_fallback / max(1, len(single))
if _pat_fallback:
    print(f"  [#10] ⚠️ {_pat_fallback}/{len(single)} imgs had no patient-id match "
          f"(treated as singleton patients — de-dup weakened for that subset).")

# fixed base / val-novel / test-novel class split
_rs = np.random.default_rng(TA_SPLIT_SEED)
_avail = [c for c in range(N_CLASSES) if len(pool[c]) >= TA_SHOT + TA_QUERY]
_perm = list(_rs.permutation(_avail))
if len(_avail) >= 5:
    TEST_NOVEL, VAL_NOVEL, BASE = sorted(_perm[:3]), sorted(_perm[3:4]), sorted(_perm[4:])
else:
    TEST_NOVEL, VAL_NOVEL, BASE = sorted(_avail), [], []
print(f"  class split (seed {TA_SPLIT_SEED}) — "
      f"base:{[TARGET_CLASSES[c] for c in BASE]} "
      f"val-novel:{[TARGET_CLASSES[c] for c in VAL_NOVEL]} "
      f"test-novel:{[TARGET_CLASSES[c] for c in TEST_NOVEL]}")

# [R13] Quantify the "no meta-training" limitation as measured numbers, not a comment.
#       With so few classes the base split cannot form N-way episodes -> 0 base episodes,
#       so φ is support-adapted at test time, never meta-trained on base tasks.
_base_can_episode = len(BASE) >= TA_WAY
META_TRAINING = {
    "n_target_classes": N_CLASSES,
    "n_base_classes": len(BASE),
    "n_val_novel_classes": len(VAL_NOVEL),
    "n_test_novel_classes": len(TEST_NOVEL),
    "way": TA_WAY,
    "base_can_form_Nway_episodes": bool(_base_can_episode),
    "theoretical_base_episodes": 0 if not _base_can_episode else int(TA_EPISODES),
    "phi_regime": "support-adapted (test-time), NOT meta-trained",
}
print(f"  [R13] meta-training budget — base classes={len(BASE)} (need ≥{TA_WAY} for "
      f"{TA_WAY}-way): {'CAN' if _base_can_episode else 'CANNOT'} form base episodes; "
      f"theoretical base episodes = {META_TRAINING['theoretical_base_episodes']}. "
      f"φ is support-adapted at test time, NOT meta-trained. CFSL numbers must be read "
      f"as a lower bound vs a properly meta-trained φ.")

INDOMAIN_OK = len(TEST_NOVEL) >= TA_WAY
if not INDOMAIN_OK:
    print(f"  ⚠️ test-novel < {TA_WAY} classes — in-domain Track A limited/skipped.")

# [#3] IMAGE-LEVEL held-out validation split, taken BEFORE balancing, so HP tuning
#      ([R12] PHI_WD) never sees an image that appears in a reported episode. `bal` is the
#      main (test) pool; `bal_val` is the disjoint validation pool.
def _image_split(items, frac, seed):
    """Return (train_keys, val_keys) with disjoint IMAGES (and disjoint patients when the
       key encodes one — a val image's patient is kept out of train entirely)."""
    rngp = np.random.default_rng(seed)
    it = list(rngp.permutation(list(items)))
    n_val = int(round(len(it) * frac))
    val = it[:n_val]
    val_pats = {_nih_patient(k) for k in val}
    train = [k for k in it[n_val:] if _nih_patient(k) not in val_pats]
    return train, val

# balance test-novel pools to a common cap (on the TRAIN split only)
bal, bal_val, bal_full = {}, {}, {}
if INDOMAIN_OK:
    _train_pool, _val_pool = {}, {}
    for c in TEST_NOVEL:
        tr, va = _image_split(pool[c], VAL_FRAC, TA_SPLIT_SEED + 50 + c)
        _train_pool[c], _val_pool[c] = tr, va
    CAP = min(len(_train_pool[c]) for c in TEST_NOVEL)
    _dr = np.random.default_rng(TA_SPLIT_SEED + 1)
    bal = {c: list(_dr.permutation(_train_pool[c])[:CAP]) for c in TEST_NOVEL}
    print(f"  balanced test-novel TRAIN pools to {CAP} imgs/class "
          f"(held out {VAL_FRAC:.0%} per class as image-disjoint validation [#3])")
    _capv = min(len(_val_pool[c]) for c in TEST_NOVEL)
    if _capv >= TA_SHOT + TA_MIN_QUERY:
        _drv = np.random.default_rng(TA_SPLIT_SEED + 2)
        bal_val = {c: list(_drv.permutation(_val_pool[c])[:_capv]) for c in TEST_NOVEL}
        print(f"  [#3] image-disjoint validation pool: {_capv} imgs/class")
    else:
        print(f"  [#3] ⚠️ validation pool too small ({_capv}/class) — WD sweep will "
              f"fall back to the default PHI_WD.")

    # [R5] coverage-controlled pool: restrict to FULLBANK so the causal / spur-only /
    #      all-patch ablation compares the SAME images (equal N per episode). Drawn from
    #      the TRAIN split so it stays disjoint from validation.
    _pool_full = {c: [k for k in bal[c] if k in FULLBANK] for c in TEST_NOVEL}
    _cap_full = min(len(_pool_full[c]) for c in TEST_NOVEL) if TEST_NOVEL else 0
    if _cap_full >= TA_SHOT + TA_MIN_QUERY:
        _drf = np.random.default_rng(TA_SPLIT_SEED + 3)
        bal_full = {c: list(_drf.permutation(_pool_full[c])[:_cap_full]) for c in TEST_NOVEL}
        print(f"  [R5] FULLBANK-restricted ablation pool: {_cap_full} imgs/class "
              f"(all present in causal ∧ spin ∧ spout)")
    else:
        print(f"  [R5] ⚠️ FULLBANK pool too small ({_cap_full}/class) — coverage-matched "
              f"ablation skipped.")

## §2 — φ transform (CFSL now · QFSL stub)
The injection point. `PhiClassical` = `512→n_qubits reduce → tanh-MLP mix → L2`. `PhiQuantum` raises until you drop the VQC in; `reduce` stays shared for a param-matched swap. Includes support-adaptation and true-TENT.

In [ ]:
# ==============================================================================
# SECTION 2 — φ TRANSFORM (CFSL now · QFSL stub) + adaptation + TENT
# ==============================================================================
class PhiClassical(nn.Module):
    """512 -> n_qubits (angle-like) -> tanh-MLP mix (VQC-analog) -> n_qubits -> L2.
       [#2] `gen` (a torch.Generator) makes the weight init DETERMINISTIC: without it,
            nn.Linear draws from the global torch RNG, so φ-adapted numbers depend on
            execution order and a [R16] resume is silently a different experiment."""
    def __init__(self, d_in=512, n_qubits=N_QUBITS, depth=PHI_DEPTH, gen=None):
        super().__init__()
        self.reduce = nn.Linear(d_in, n_qubits)
        blk = []
        for _ in range(depth): blk += [nn.Linear(n_qubits, n_qubits), nn.Tanh()]
        self.mix = nn.Sequential(*blk)
        if gen is not None: self._seeded_init(gen)
    def _seeded_init(self, gen):
        # Reproduce nn.Linear's default init distribution (weight & bias ~ U(-1/√fan_in,
        # +1/√fan_in)) but from a supplied generator, so it is fully reproducible.
        with torch.no_grad():
            for m in self.modules():
                if isinstance(m, nn.Linear):
                    b = 1.0 / math.sqrt(m.weight.shape[1])
                    m.weight.uniform_(-b, b, generator=gen)
                    if m.bias is not None: m.bias.uniform_(-b, b, generator=gen)
    def forward(self, x):
        z = torch.tanh(self.reduce(x))          # bounded, mimics angle encoding
        return F.normalize(self.mix(z), dim=-1)

class PhiQuantum(nn.Module):
    """QFSL drop-in — fill when the quantum cell lands. Keep `reduce` identical."""
    def __init__(self, d_in=512, n_qubits=N_QUBITS, depth=PHI_DEPTH):
        super().__init__()
        self.reduce = nn.Linear(d_in, n_qubits)
        # ================= QUANTUM INJECTION POINT =================
        # pennylane: AngleEmbedding(tanh(reduce(x))) -> StronglyEntanglingLayers(depth)
        #            -> [<Z_0>, ..., <Z_{n-1}>] -> F.normalize(...)
        raise NotImplementedError(
            "QFSL φ_q is a stub. Provide the VQC in PhiQuantum.forward and set "
            "RUN_QUANTUM=True. Keep `reduce` shared with PhiClassical for a "
            "parameter-matched comparison. Load the saved episodes/*.pt (in-domain AND "
            "cross-domain, incl. anchor tensors) and reuse the classical baselines in "
            "fsl_cfsl_qfsl_results.json for comparison.")

def build_phi(mode="classical", init_seed=None):
    """[#2] Pass `init_seed` (thread the SAME seed*1e6+episode used for adapt_phi's rng)
       to make φ's weight init reproducible and independent of execution order."""
    gen = None
    if init_seed is not None:
        gen = torch.Generator(device="cpu").manual_seed(int(init_seed) % (2**63 - 1))
    if mode == "quantum":
        return PhiQuantum().to(Device)                  # stub raises; init_seed N/A
    return PhiClassical(gen=gen).to(Device)

# [7] transparent parameter accounting (reduce is shared scaffolding; compare `mix`)
def _param_report():
    p = PhiClassical()
    nred = sum(t.numel() for t in p.reduce.parameters())
    nmix = sum(t.numel() for t in p.mix.parameters())
    vqc  = 3 * N_QUBITS * PHI_DEPTH  # StronglyEntanglingLayers rotation angles
    print(f"  [7] φ params — reduce(shared): {nred} | classical mix: {nmix} | "
          f"VQC angles (future): {vqc}. Comparison is on the `mix`/VQC block; "
          f"`reduce` is identical scaffolding in both.")
_param_report()

def _protos(z, y, way):
    return F.normalize(torch.stack([z[y == c].mean(0) for c in range(way)]), dim=-1)

def _transductive_refine(P, qz, way):
    """[11][12] order-invariant, margin-gated, temperature-free transductive update.
       Shared by proto_cosine and eval_phi so the φ rows can KEEP transduction [R1]."""
    sim = qz @ P.T
    top2 = torch.topk(sim, min(2, way), dim=1).values
    margin = top2[:, 0] - top2[:, 1] if way > 1 else top2[:, 0]
    arg = sim.argmax(1)
    newP = P.clone()
    for c in range(way):
        hit = (arg == c) & (margin > TA_QT_MARGIN)
        if hit.any():
            newP[c] = F.normalize(0.9 * P[c] + 0.1 * qz[hit].mean(0), dim=-1)
    return newP

def adapt_phi(phi, sup_x, sup_y, way, shot, rng=None, wd=None):
    """Per-episode support adaptation via pseudo-support/pseudo-query split.
       shot<2 -> cannot split -> φ returned frozen (honest). [9][19]
       [R2] `rng` is threaded per-episode by the caller so the held-out pseudo-query
            truly varies across episodes/seeds (no more default_rng(0) on every call).
       [R12] `wd` overrides PHI_WD for the weight-decay sensitivity sweep."""
    if shot < 2: return phi
    if rng is None:
        raise ValueError("adapt_phi requires a per-episode rng (fix [R2]); "
                         "callers must thread np.random.default_rng(seed, episode).")
    wd = PHI_WD if wd is None else wd
    ps, pq = [], []
    for c in range(way):
        ids = (sup_y == c).nonzero(as_tuple=True)[0].tolist()
        held = int(rng.choice(ids))           # [19][R2] random held-out pseudo-query
        pq.append(held); ps += [i for i in ids if i != held]
    ps = torch.tensor(ps, device=sup_x.device); pq = torch.tensor(pq, device=sup_x.device)
    tgt = torch.arange(way, device=sup_x.device)
    steps = _phi_steps(shot)                   # [9] scale steps with K
    phi.train(); opt = torch.optim.AdamW(phi.parameters(), lr=PHI_LR, weight_decay=wd)
    for _ in range(steps):
        z = phi(sup_x)
        P = _protos(z[ps], sup_y[ps], way)
        loss = F.cross_entropy((z[pq] @ P.T) / ADAPT_TEMP, tgt)
        opt.zero_grad(); loss.backward(); opt.step()
    phi.eval(); return phi

def tent_phi(phi, sup_x, sup_y, qry_x, way, steps=TENT_STEPS, lr=TENT_LR):
    """[20] True TENT: prototypes fixed once from support; minimise query-logit entropy.
       [R3] Uses SHARP TENT_TEMP (not COS_TEMP=1.0). Cosine sims live in [-1,1]; at
            temperature 1 the softmax is near-uniform -> near-max entropy -> vanishing
            entropy gradient -> TENT cannot move. TENT_TEMP=0.1 restores a real gradient.
       DISCLOSURE (#minor): φ is adapted on the SAME query batch it is then scored on —
            standard transductive TTA, but a genuine test-time (not train-time) update.
            Stated here so it need not be reverse-engineered from the call site."""
    with torch.no_grad():
        P = _protos(phi(sup_x), sup_y, way).detach()   # fixed anchor prototypes
    phi.train(); opt = torch.optim.Adam(phi.parameters(), lr=lr)
    for _ in range(steps):
        p = torch.softmax((phi(qry_x) @ P.T) / TENT_TEMP, dim=1).clamp(1e-6, 1 - 1e-6)
        ent = -(p * p.log()).sum(1).mean()
        opt.zero_grad(); ent.backward(); opt.step()
    phi.eval(); return phi

@torch.no_grad()
def eval_phi(phi, sup_x, sup_y, qry_x, qry_y, way, anchor=None, transductive=False,
            alpha=None):
    """[R4] Anchoring is done in the RAW 512-d space and projected ONCE through φ:
            NIH source prototypes are NEVER pushed through a CheXpert-fit projection
            into an unrelated coordinate system. We blend raw 512-d prototypes, then
            apply φ a single time so query and prototypes share φ's output geometry.
       [R1] `transductive` lets the TTDA ladder KEEP query-time refinement while adding
            φ, so φ is the only new factor (true one-change-per-row).
       `alpha` is threaded explicitly (no global mutation) so the [R11]-chosen blend is
            passed in rather than read from a mutable global (round-3 #minor)."""
    a = CX_ANCHOR_ALPHA if alpha is None else alpha
    qz = F.normalize(phi(qry_x), dim=-1)
    if anchor is not None:
        # blend in 512-d, then a single φ projection (fix [R4])
        sup_proto_512 = F.normalize(
            torch.stack([sup_x[sup_y == c].mean(0) for c in range(way)]), dim=-1)
        blended_512 = F.normalize(
            a * F.normalize(anchor, dim=-1) + (1 - a) * sup_proto_512, dim=-1)
        P = F.normalize(phi(blended_512), dim=-1)
    else:
        P = _protos(phi(sup_x), sup_y, way)
    if transductive:
        P = _transductive_refine(P, qz, way)
    scores = (qz @ P.T).cpu().numpy()                  # [10] raw cosine
    return _metrics(qry_y, scores)

## §3 — Zero-param metric methods
Cosine prototype (+ spur cleaning, order-invariant transduction, anchoring), genuine-Euclidean ProtoNet, RelationNet. Reported AUROC uses **raw cosine** (temperature-free).

In [ ]:
# ==============================================================================
# SECTION 3 — ZERO-PARAM METRIC METHODS + shared metric + statistics
# ==============================================================================
def _metrics(yt, scores):
    """[10] scores are per-class similarities (queries × way). AUROC on raw scores.
       [14] AUROC and F1 share the SAME per-episode validity: if <2 classes present,
            all are nan together.
       [R9] Returns per-class AUROC (nan for absent classes) alongside the macro mean.
       Returns (macro_auroc, macro_f1, per_class_auroc_list[way])."""
    yt = np.asarray(yt); scores = np.asarray(scores)
    way = scores.shape[1]
    present = [c for c in range(way) if 0 < (yt == c).sum() < len(yt)]
    if len(present) < 2:
        return float("nan"), float("nan"), [float("nan")] * way
    per_class = [float("nan")] * way
    for c in present:
        per_class[c] = float(roc_auc_score((yt == c).astype(int), scores[:, c]))
    au = float(np.mean([per_class[c] for c in present]))
    f1 = float(f1_score(yt, scores.argmax(1), average="macro"))
    return au, f1, per_class

# ---- statistics helpers (round-2) ------------------------------------------
def _bootstrap_ci(a, n=None, seed=BOOT_SEED):
    """[R7] Flat bootstrap percentile 95% CI (assumes i.i.d. episodes). AUROC is
       bounded/asymmetric — the normal approximation is inappropriate near 0.5/1.0."""
    a = np.asarray([x for x in a if not np.isnan(x)], dtype=float)
    if len(a) < 2:
        return float("nan"), float("nan")
    n = BOOTSTRAP_N if n is None else n
    rng = np.random.default_rng(seed)
    idx = rng.integers(0, len(a), size=(n, len(a)))
    boots = a[idx].mean(axis=1)
    return float(np.percentile(boots, 2.5)), float(np.percentile(boots, 97.5))

def _bootstrap_ci_cluster(au_by_seed, n=None, seed=BOOT_SEED):
    """[#5] Cluster (block) bootstrap: episodes across seeds are NOT i.i.d. ([R8] shows
       nonzero image overlap), so a flat episode bootstrap under-covers. Resample whole
       SEEDS with replacement, then episodes within each chosen seed, and pool. Falls back
       to the flat bootstrap when there is only one seed (e.g. validation runs)."""
    clusters = [np.asarray([x for x in s if not np.isnan(x)], dtype=float)
                for s in au_by_seed]
    clusters = [c for c in clusters if len(c) > 0]
    if not clusters:
        return float("nan"), float("nan")
    if len(clusters) == 1:
        return _bootstrap_ci(clusters[0], n, seed)
    n = BOOTSTRAP_N if n is None else n
    rng = np.random.default_rng(seed); S = len(clusters); boots = np.empty(n)
    for b in range(n):
        chosen = rng.integers(0, S, size=S)
        vals = [clusters[ci][rng.integers(0, len(clusters[ci]), size=len(clusters[ci]))]
                for ci in chosen]
        boots[b] = np.concatenate(vals).mean()
    return float(np.percentile(boots, 2.5)), float(np.percentile(boots, 97.5))

def holm_bonferroni(pvals):
    """[#4] Holm–Bonferroni step-down adjustment across a family of p-values. Returns
       adjusted p-values aligned to the input order (nan stays nan). Controls FWER so the
       ~dozens of paired Wilcoxon tests don't yield ~1–2 spurious stars by chance."""
    idx = [i for i, p in enumerate(pvals) if p is not None and p == p]
    m = len(idx); adj = [float("nan")] * len(pvals)
    order = sorted(idx, key=lambda i: pvals[i]); prev = 0.0
    for rank, i in enumerate(order):
        a = min(1.0, (m - rank) * pvals[i]); a = max(a, prev); prev = a
        adj[i] = a
    return adj

def paired_wilcoxon(pairs_a, pairs_b):
    """[R6] Paired Wilcoxon signed-rank on per-episode AUROC over episodes valid in
       BOTH methods. Returns a two-sided p-value (nan if too few pairs)."""
    keys = sorted(set(pairs_a) & set(pairs_b))
    a = np.array([pairs_a[k] for k in keys], dtype=float)
    b = np.array([pairs_b[k] for k in keys], dtype=float)
    m = ~(np.isnan(a) | np.isnan(b))
    a, b = a[m], b[m]
    if len(a) < WILCOXON_MIN_N:
        return float("nan")
    if np.allclose(a, b):
        return 1.0
    try:
        return float(wilcoxon(a, b, alternative="two-sided").pvalue)
    except Exception:
        return float("nan")

def _stars(p):
    if p is None or (isinstance(p, float) and np.isnan(p)): return "   "
    if p < 0.001: return "***"
    if p < 0.01:  return "** "
    if p < 0.05:  return "*  "
    return "ns "

@torch.no_grad()
def proto_cosine(sup_x, sup_y, qry_x, qry_y, way, transductive=False,
                 anchor=None, clean=False, alpha=None):
    # NOTE (#8): `clean` here subtracts the DATASET-LEVEL mean `spout_dir` (used
    # cross-domain, where CheXpert has no per-image spur bank). This is NOT the same
    # operation as the in-domain `causal_full` variant, which subtracts each image's OWN
    # sig_spin/sig_spout (per-sample). The two are documented as distinct in the header.
    a = CX_ANCHOR_ALPHA if alpha is None else alpha
    if clean:
        if spout_dir is None:                          # [3] no silent no-op
            raise RuntimeError("clean=True requested but spur_out bank is empty; "
                               "cannot perform spur_out cleaning.")
        sd = spout_dir.to(sup_x.device)
        sup_x = F.normalize(sup_x - TA_BETA * sd, dim=-1)
        qry_x = F.normalize(qry_x - TA_BETA * sd, dim=-1)
        if anchor is not None: anchor = F.normalize(anchor - TA_BETA * sd, dim=-1)
    P = _protos(sup_x, sup_y, way)
    if anchor is not None:
        P = F.normalize(a * F.normalize(anchor, dim=-1) + (1 - a) * P, dim=-1)
    if transductive:                                    # [11][12] shared helper
        P = _transductive_refine(P, qry_x, way)
    scores = (qry_x @ P.T).cpu().numpy()                # [10] raw cosine
    return _metrics(qry_y, scores)

@torch.no_grad()
def proto_euclidean(sup_x_raw, sup_y, qry_x_raw, qry_y, way):
    """[13] ProtoNet on UNNORMALISED features -> genuine Euclidean geometry,
       an independent family from cosine (not a monotone re-parameterisation)."""
    P = torch.stack([sup_x_raw[sup_y == c].mean(0) for c in range(way)])
    d = -((qry_x_raw.unsqueeze(1) - P.unsqueeze(0)) ** 2).sum(-1)  # neg sq-Euclidean
    return _metrics(qry_y, d.cpu().numpy())

def linear_probe(sup_x, sup_y, qry_x, qry_y, way):
    """[R10] Discriminative calibration point: L2-logistic regression on frozen support
       features. Anchors the cosine-prototype claims and the K-sweep (LR has a known
       sample-efficiency curve)."""
    Xs = sup_x.detach().cpu().numpy()
    ys = sup_y.detach().cpu().numpy() if torch.is_tensor(sup_y) else np.asarray(sup_y)
    if len(np.unique(ys)) < 2:
        return float("nan"), float("nan"), [float("nan")] * way
    clf = LogisticRegression(max_iter=1000, C=1.0)
    clf.fit(Xs, ys)
    proba = clf.predict_proba(qry_x.detach().cpu().numpy())
    sc = np.zeros((qry_x.shape[0], way), dtype=float)
    for j, cl in enumerate(clf.classes_):
        sc[:, int(cl)] = proba[:, j]
    return _metrics(qry_y, sc)

def relation_net(sup_x, sup_y, qry_x, qry_y, way):      # RelationNet baseline
    d = sup_x.shape[1]
    rel = nn.Sequential(nn.Linear(2 * d, 128), nn.ReLU(), nn.Linear(128, 1)).to(sup_x.device)
    opt = torch.optim.Adam(rel.parameters(), lr=1e-2)
    P = torch.stack([sup_x[sup_y == c].mean(0) for c in range(way)])
    Xs, ys = [], []
    for i in range(sup_x.shape[0]):
        for cc in range(way):
            Xs.append(torch.cat([sup_x[i], P[cc]])); ys.append(1.0 if sup_y[i] == cc else 0.0)
    Xs = torch.stack(Xs); ys = torch.tensor(ys, device=sup_x.device)
    pos_w = torch.tensor([(way - 1.0)], device=sup_x.device)  # [22] balance 1:(way-1)
    rel.train()
    for _ in range(40):
        opt.zero_grad()
        F.binary_cross_entropy_with_logits(rel(Xs).squeeze(-1), ys, pos_weight=pos_w).backward()
        opt.step()
    rel.eval()
    with torch.no_grad():
        sc = []
        for j in range(qry_x.shape[0]):
            pr = torch.stack([torch.cat([qry_x[j], P[cc]]) for cc in range(way)])
            sc.append(rel(pr).squeeze(-1).cpu().numpy())   # [10] raw relation scores
    r = _metrics(qry_y, np.stack(sc))
    del rel, opt
    return r

## §4 — Feature variants + episode sampling
`causal / +spur_in / +spur_out / causal_full / all-patch`. `gen_episodes` returns true `ways`, is patient-disjoint, enforces the full query budget, and reports drops.

In [ ]:
# ==============================================================================
# SECTION 4 — FEATURE VARIANTS + EPISODE SAMPLING
# ==============================================================================
def feat_vec(key, variant):
    """variant: all | causal | causal_spin | causal_spout | causal_full | spur_only
       [R21] spur_only = mean(spin, spout) with NO causal channel (the "without-causal,
             spurious-patches-only concatenated" ablation)."""
    if variant == "spur_only":
        parts = []
        if key in sig_spin:  parts.append(sig_spin[key])
        if key in sig_spout: parts.append(sig_spout[key])
        if not parts:                       # should not happen on FULLBANK pool
            return F.normalize(torch.zeros_like(sig_causal[key]) + 1e-6, dim=0)
        return F.normalize(torch.stack(parts).mean(0), dim=0)
    v = (sig_all[key] if variant == "all" else sig_causal[key]).clone()
    if variant in ("causal_spin", "causal_full") and key in sig_spin:
        v = v - TA_BETA * sig_spin[key]
    if variant in ("causal_spout", "causal_full") and key in sig_spout:
        v = v - TA_BETA * sig_spout[key]
    return F.normalize(v, dim=0)

def stack_feats(keys, variant):
    return torch.stack([feat_vec(k, variant) for k in keys]).to(Device)

def stack_raw(keys):
    return torch.stack([sig_causal_raw[k].clone() for k in keys]).to(Device)  # [13]

def gen_episodes(seed, classes, pools, K, patient_of=None):
    """Return (episodes, n_dropped). Each episode = (sk, sy, qk, qy, ways) where
       ways[label] is the TRUE class id -> fixes anchor alignment [1].
       [8][#1] patient-disjoint at the EPISODE level (not per-class): a two-pass build
               collects support patients across ALL classes first, then admits queries
               only for patients that are in NO class's support. This closes the
               cross-class leakage channel (patient P in class-A support AND class-B
               query) that per-class `used_pat` left open.
       [16] enforce len(qry) >= TA_MIN_QUERY.  [17] count drops."""
    rng = np.random.default_rng(seed); eps = []; dropped = 0
    for _ in range(TA_EPISODES):
        ways = list(rng.choice(classes, min(TA_WAY, len(classes)), replace=False))
        sk, sy, qk, qy = [], [], [], []
        ok = True
        if patient_of is not None:
            # PASS 1 — pick support with EPISODE-GLOBAL patient disjointness [#1]
            used_pat, per_class_items, sup_by_lbl = set(), {}, {}
            for lbl, c in enumerate(ways):
                items = list(pools[c]); rng.shuffle(items); per_class_items[lbl] = items
                sup, i = [], 0
                while len(sup) < K and i < len(items):
                    p = patient_of[items[i]]
                    if p not in used_pat: sup.append(items[i]); used_pat.add(p)
                    i += 1
                if len(sup) < K: ok = False; break
                sup_by_lbl[lbl] = sup
            # PASS 2 — queries exclude EVERY support patient (across all classes) [#1]
            if ok:
                for lbl, c in enumerate(ways):
                    qry = [it for it in per_class_items[lbl]
                           if patient_of[it] not in used_pat][:TA_QUERY]
                    if len(qry) < TA_MIN_QUERY: ok = False; break
                    sk += sup_by_lbl[lbl]; sy += [lbl] * len(sup_by_lbl[lbl])
                    qk += qry;             qy += [lbl] * len(qry)
        else:
            for lbl, c in enumerate(ways):
                p = list(rng.permutation(list(pools[c])))
                sup, qry = p[:K], p[K:K + TA_QUERY]
                if len(sup) < K or len(qry) < TA_MIN_QUERY: ok = False; break
                sk += list(sup); sy += [lbl] * len(sup); qk += list(qry); qy += [lbl] * len(qry)
        if ok: eps.append((sk, sy, qk, qy, [int(w) for w in ways]))
        else:  dropped += 1
    return eps, dropped

def episode_overlap_fraction(episodes_by_seed):
    """[R8] Fraction of images shared across ALL seeds (|∩| / |∪|) over the pooled
       support+query image sets per seed. Quantifies how independent the seed replicates
       actually are — a reader cannot assess the pooled-CI independence claim without it."""
    per_seed_imgs = {}
    for sd, eps in episodes_by_seed.items():
        s = set()
        for (sk, sy, qk, qy, ways) in eps:
            s.update(sk); s.update(qk)
        per_seed_imgs[sd] = s
    sets = [s for s in per_seed_imgs.values() if s]
    if len(sets) < 2:
        return {"n_seeds": len(sets), "overlap_fraction": 0.0}
    union = set().union(*sets); inter = set(sets[0]).intersection(*sets[1:])
    return {"n_seeds": len(sets),
            "overlap_fraction": (len(inter) / len(union)) if union else 0.0,
            "n_union_imgs": len(union), "n_shared_imgs": len(inter)}

## §5 — In-domain runner (NIH, Track A)
Headline @K=5, the K∈{1..5} sample-efficiency sweep, and the one-change-per-row TTDA ladder. One truly-frozen φ is shared across all episodes.

In [ ]:
# ==============================================================================
# SECTION 5 — IN-DOMAIN RUNNER (NIH, Track A)
# ==============================================================================
OUT_OVERLAP = {}          # [R8] seed image-overlap fractions, logged to JSON

# [R16] per-seed checkpointing --------------------------------------------------
_CKPT_PREFIX = ("dry" if DRY_RUN else "full")
_CKPT_REGISTRY = {}       # [#minor] sanitized-path -> raw tag, to catch tag collisions
def _ckpt_path(tag):
    safe = re.sub(r"[^A-Za-z0-9._-]", "_", f"{_CKPT_PREFIX}__{tag}")
    return f"{CKPT_DIR}/{safe}.pt"
def _register_ckpt(tag):
    if not tag: return
    p = _ckpt_path(tag)
    if p in _CKPT_REGISTRY and _CKPT_REGISTRY[p] != tag:
        raise ValueError(f"[#minor] checkpoint tag collision: '{tag}' and "
                         f"'{_CKPT_REGISTRY[p]}' both sanitize to {os.path.basename(p)}")
    _CKPT_REGISTRY[p] = tag
def _ckpt_load(tag):
    p = _ckpt_path(tag)
    if tag and os.path.isfile(p):
        try: return torch.load(p, map_location="cpu", weights_only=False)
        except Exception: return {}
    return {}
def _ckpt_save(tag, obj):
    if tag:
        try: torch.save(obj, _ckpt_path(tag))
        except Exception as e: print(f"    [R16] ckpt save failed for {tag}: {e}")

def _agg(au_s, f1_s, au_all, pc_named=None, au_by_seed=None):
    """Per-seed means + pooled CI + CLUSTER bootstrap CI + per-class AUROC. [28][R7][R9][#5]"""
    out = {"auroc_mean": float(np.mean(au_s)) if au_s else float("nan"),
           "auroc_std":  float(np.std(au_s)) if au_s else 0.0,
           "f1_mean":    float(np.mean(f1_s)) if f1_s else float("nan"),
           "f1_std":     float(np.std(f1_s)) if f1_s else 0.0,
           "n_episodes": int(len(au_all))}
    if au_all:
        a = np.asarray(au_all, dtype=float)
        out["auroc_pooled_ci95"] = float(1.96 * a.std(ddof=1) / max(1, np.sqrt(len(a))))  # legacy
        # [#5] cluster bootstrap (resample seeds then episodes) is the REPORTED CI;
        #      falls back to flat bootstrap when only one seed is present.
        lo, hi = (_bootstrap_ci_cluster(au_by_seed) if au_by_seed else _bootstrap_ci(a))
        out["auroc_boot_ci95_lo"] = lo; out["auroc_boot_ci95_hi"] = hi
    if pc_named:
        out["per_class_auroc"] = {k: float(np.mean(v)) for k, v in pc_named.items() if v}
    return out

def run_method_over_episodes(episodes_by_seed, fn, ckpt_tag=None):
    """Runs `fn` over every episode. Threads a UNIQUE per-episode rng AND integer seed
       [R2][#2] (the int seeds both the pseudo-query draw and φ's weight init), accumulates
       per-episode AUROC pairs for Wilcoxon [R6], per-class AUROC [R9], per-seed clusters
       for the cluster bootstrap [#5], wall-clock [R17], and checkpoints each seed [R16]."""
    _register_ckpt(ckpt_tag)
    t0 = time.perf_counter()
    ck = _ckpt_load(ckpt_tag)
    au_s, f1_s, au_all, pairs, pc_named, au_by_seed = [], [], [], {}, _dd(list), []
    for sd, eps in episodes_by_seed.items():
        skey = str(sd)
        if skey in ck:
            rec = ck[skey]
        else:
            aus, f1s, ppairs, pcs = [], [], {}, _dd(list)
            for ei, (sk, sy, qk, qy, ways) in enumerate(eps):
                ep_seed = int(sd) * 1_000_003 + ei                # [R2][#2] unique int
                ep_rng = np.random.default_rng(ep_seed)
                au, f1, pc = fn(sk, torch.tensor(sy, dtype=torch.long),
                                qk, np.array(qy), ways, ep_rng, ep_seed)
                if not (np.isnan(au) or np.isnan(f1)):
                    aus.append(au); f1s.append(f1); ppairs[f"{sd}_{ei}"] = au
                    for lbl, v in enumerate(pc):
                        if not np.isnan(v): pcs[TARGET_CLASSES[ways[lbl]]].append(v)
            rec = {"aus": aus, "f1s": f1s, "pairs": ppairs,
                   "pcs": {k: list(v) for k, v in pcs.items()}}
            if ckpt_tag: ck[skey] = rec; _ckpt_save(ckpt_tag, ck)
        if rec["aus"]:
            au_s.append(np.mean(rec["aus"])); f1_s.append(np.mean(rec["f1s"]))
            au_by_seed.append(rec["aus"])
        au_all += rec["aus"]; pairs.update(rec["pairs"])
        for k, v in rec["pcs"].items(): pc_named[k] += v
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()   # [23]
    r = _agg(au_s, f1_s, au_all, pc_named, au_by_seed)
    r["_pairs"] = pairs                                 # underscore -> stripped from JSON
    r["wall_sec"] = float(time.perf_counter() - t0)     # [R17]
    return r

# method closures (signature: sk, sy, qk, qy, ways, rng, ep_seed) ------------
def M_proto_euclid():
    def fn(sk, sy, qk, qy, ways, rng, ep_seed):
        return proto_euclidean(stack_raw(sk), sy.to(Device), stack_raw(qk), qy, TA_WAY)
    return fn
def M_linear(variant="causal_full"):                    # [R10] logistic-regression probe
    def fn(sk, sy, qk, qy, ways, rng, ep_seed):
        return linear_probe(stack_feats(sk, variant), sy.to(Device),
                            stack_feats(qk, variant), qy, TA_WAY)
    return fn
def M_relation(variant="all"):
    def fn(sk, sy, qk, qy, ways, rng, ep_seed):
        return relation_net(stack_feats(sk, variant), sy.to(Device),
                            stack_feats(qk, variant), qy, TA_WAY)
    return fn
def M_cosine(variant, transductive=False):
    def fn(sk, sy, qk, qy, ways, rng, ep_seed):
        return proto_cosine(stack_feats(sk, variant), sy.to(Device),
                            stack_feats(qk, variant), qy, TA_WAY, transductive=transductive)
    return fn
def M_phi(variant, mode="classical", adapt=True, tent=False, transductive=False,
          K=TA_SHOT, frozen_phi=None, wd=None):
    def fn(sk, sy, qk, qy, ways, rng, ep_seed):
        sx, qx = stack_feats(sk, variant), stack_feats(qk, variant)
        syd = sy.to(Device)
        if frozen_phi is not None:                       # [4] one fixed init, cloned
            phi = copy.deepcopy(frozen_phi)
        else:
            phi = build_phi(mode, init_seed=ep_seed)     # [#2] reproducible init
            if adapt: phi = adapt_phi(phi, sx, syd, TA_WAY, K, rng=rng, wd=wd)  # [R2][R12]
            if tent:  phi = tent_phi(phi, sx, syd, qx, TA_WAY)
        r = eval_phi(phi, sx, syd, qx, qy, TA_WAY, transductive=transductive)   # [R1]
        del phi
        return r
    return fn
def M_phi_wd(wd, K=TA_SHOT, variant="causal_full"):     # [R12] WD-sweep closure
    def fn(sk, sy, qk, qy, ways, rng, ep_seed):
        sx, qx = stack_feats(sk, variant), stack_feats(qk, variant); syd = sy.to(Device)
        phi = adapt_phi(build_phi("classical", init_seed=ep_seed), sx, syd, TA_WAY, K,
                        rng=rng, wd=wd)
        r = eval_phi(phi, sx, syd, qx, qy, TA_WAY); del phi; return r
    return fn

# [4][#2] one truly-frozen classical φ with a REPRODUCIBLE init, fixed across all episodes.
# NOTE (#9): this φ is UNTRAINED — a fixed RANDOM projection, not a pretrained/meta-trained
# encoder. It is a sanity floor ("does support-adaptation beat a random φ?"), not an
# adapted-vs-pretrained comparison (there is no pretrained φ in this codebase; see [R13]).
FROZEN_PHI = build_phi("classical", init_seed=20240717); FROZEN_PHI.eval()

# ---- pretty-printers with per-class + Holm-corrected significance -----------
def _fmt_ci(r):
    lo, hi = r.get("auroc_boot_ci95_lo"), r.get("auroc_boot_ci95_hi")
    if lo is None or hi is None or (isinstance(lo, float) and np.isnan(lo)):
        return "        -        "
    return f"[{lo:.3f},{hi:.3f}]"

def _print_perclass(res):
    names = sorted({k for r in res.values() for k in r.get("per_class_auroc", {})})
    if not names: return
    print("\n  Per-class AUROC [R9]:")
    print("    " + " " * 32 + "".join(f"{n[:7]:>9}" for n in names))
    for nm, r in res.items():
        pc = r.get("per_class_auroc", {})
        print(f"    {nm:<32}" + "".join(
            f"{pc[n]:>9.3f}" if n in pc else f"{'-':>9}" for n in names))

def _print_deltas(res, ref_name, adjacent=False):
    """Δ + paired Wilcoxon, HOLM-CORRECTED across the family [R6][#4]. adjacent=True -> vs
       previous row (ladders); else vs `ref_name` (headline/ablation)."""
    items = list(res.items())
    ref = res.get(ref_name, items[0][1])
    rows, pv, prev = [], [], None
    for nm, r in items:
        base = prev if (adjacent and prev is not None) else ref
        if base is None: base = r
        p = (paired_wilcoxon(base.get("_pairs", {}), r.get("_pairs", {}))
             if base is not r else float("nan"))
        rows.append((nm, r, r["auroc_mean"] - base["auroc_mean"], p)); pv.append(p); prev = r
    adj = holm_bonferroni(pv)
    fam = sum(1 for p in pv if p == p)
    tgt = "adjacent rows" if adjacent else f"vs {ref_name}"
    print(f"\n  Δ + paired Wilcoxon ({tgt}); Holm-corrected over family of {fam} tests "
          f"[R6][#4]  (* p<.05  ** p<.01  *** p<.001):")
    print(f"    {'method':<32}{'AUROC':>9}{'Δ':>9}   raw  Holm")
    for (nm, r, d, p), pa in zip(rows, adj):
        print(f"    {nm:<32}{r['auroc_mean']:>9.4f}{d:>+9.4f}   {_stars(p)}{_stars(pa)}"
              f"  p={'nan' if p != p else f'{p:.1e}'}")

def _print_table(res, title):
    print(f"\n  {'Method':<32}{'AUROC':>16}{'clust-CI95':>19}{'macro-F1':>13}{'sec':>8}")
    for nm, r in res.items():
        print(f"  {nm:<32}{r['auroc_mean']:>8.4f}±{r['auroc_std']:.3f}"
              f"{_fmt_ci(r):>19}{r['f1_mean']:>7.3f}±{r['f1_std']:.3f}"
              f"{r.get('wall_sec',0):>8.1f}")

# ---- [R18] pre-flight sanity check ------------------------------------------
def _sanity_check():
    print("=" * 70, "\n[R18] Pre-flight sanity check (asserts before the full run)\n", "=" * 70)
    if not INDOMAIN_OK:
        print("  in-domain not OK — sanity check skipped."); return
    eps = gen_episodes(0, TEST_NOVEL, bal, TA_SHOT, patient_of=nih_patient_of)[0][:SANITY_EPISODES]
    assert eps, "no sanity episodes formed"
    for (sk, sy, qk, qy, ways) in eps:
        assert set(sk).isdisjoint(set(qk)), "support/query image keys overlap"
        # [#1] PATIENT-level disjointness across the WHOLE episode (not just per class)
        sp = {nih_patient_of.get(k, k) for k in sk}
        qp = {nih_patient_of.get(k, k) for k in qk}
        assert sp.isdisjoint(qp), "patient leakage: a support patient appears in query"
        assert len(ways) == TA_WAY, f"ways length {len(ways)} != TA_WAY {TA_WAY}"
    checks = {"cosine": M_cosine("causal_full"),
              "phi-adapt": M_phi("causal_full", adapt=True, K=TA_SHOT),
              "linear-probe": M_linear()}
    for nm, fn in checks.items():
        for ei, (sk, sy, qk, qy, ways) in enumerate(eps):
            au, f1, pc = fn(sk, torch.tensor(sy, dtype=torch.long),
                            qk, np.array(qy), ways, np.random.default_rng(ei), ei)
            assert isinstance(au, float), f"{nm}: AUROC not float"
            assert np.isnan(au) or (0.0 <= au <= 1.0), f"{nm}: AUROC {au} out of [0,1]"
            assert len(pc) == TA_WAY, f"{nm}: per-class length {len(pc)} != {TA_WAY}"
    # [#2] φ init reproducibility: same init_seed -> identical weights
    a = build_phi("classical", init_seed=123); b = build_phi("classical", init_seed=123)
    assert all(torch.equal(pa.detach().cpu(), pb.detach().cpu())
               for pa, pb in zip(a.parameters(), b.parameters())), \
        "φ init not reproducible for a fixed init_seed"
    print("  ✅ sanity checks passed (image+patient disjoint support/query, ways length, "
          "finite AUROC∈[0,1], reproducible φ init).")

# ---- [R12] weight-decay sensitivity sweep on IMAGE-held-out validation ------
def indomain_wd_sweep():
    """Returns the chosen PHI_WD (THREADED explicitly to callers — no global mutation).
       Tunes on `bal_val`, an IMAGE-disjoint pool never seen in a reported episode [#3]."""
    print("=" * 70, "\n[R12] φ weight-decay sweep (image-held-out validation)\n", "=" * 70)
    if not (INDOMAIN_OK and bal_val):
        print("  no image-disjoint validation pool — keeping default PHI_WD.")
        return {"chosen_wd": PHI_WD, "note": "skipped (no validation pool)"}
    veps = {}
    for vs in (7001, 7002):
        veps[vs] = gen_episodes(vs, TEST_NOVEL, bal_val, TA_SHOT, patient_of=nih_patient_of)[0]
    table, best = {}, None
    for wd in PHI_WD_SWEEP:
        r = run_method_over_episodes(veps, M_phi_wd(wd), ckpt_tag=f"wdsweep_wd{wd}")
        table[str(wd)] = r["auroc_mean"]
        print(f"    WD={wd:<8}  val-AUROC={r['auroc_mean']:.4f}")
        if best is None or (r["auroc_mean"] == r["auroc_mean"] and r["auroc_mean"] > best[1]):
            best = (wd, r["auroc_mean"])
    chosen = best[0]
    print(f"  → chosen PHI_WD = {chosen} (best val AUROC={best[1]:.4f}); threaded to φ rows. "
          f"Holdout is IMAGE-disjoint (class-level impossible: 1 val-novel class [R13]).")
    return {"sweep": table, "chosen_wd": chosen,
            "holdout": "image-disjoint validation pool [#3] (val seeds 7001/7002)"}

def indomain_headline(chosen_wd=None):
    print("=" * 70, "\n[5] IN-DOMAIN NIH — Track-A headline (K=5)\n", "=" * 70)
    wd = PHI_WD if chosen_wd is None else chosen_wd
    eps, ndrop = {}, 0
    for sd in TA_SEEDS:
        e, d = gen_episodes(sd, TEST_NOVEL, bal, TA_SHOT, patient_of=nih_patient_of)
        eps[sd] = e; ndrop += d
    n_ok = sum(len(e) for e in eps.values())
    print(f"  episodes kept: {n_ok} across {len(TA_SEEDS)} seeds (dropped {ndrop}) [17]")
    OUT_OVERLAP["indomain_K5"] = episode_overlap_fraction(eps)   # [R8]
    print(f"  [R8] seed image-overlap fraction (K=5): "
          f"{OUT_OVERLAP['indomain_K5']['overlap_fraction']:.4f} "
          f"(cluster bootstrap accounts for this [#5])")
    methods = {
        "Linear probe (LR, baseline)":    M_linear("causal_full"),
        "ProtoNet (Euclidean, baseline)": M_proto_euclid(),
        "RelationNet (baseline)":         M_relation("all"),
        "All-patch proto":                M_cosine("all"),
        "Ours-minimal (causal)":          M_cosine("causal"),
        "+ spur_in":                      M_cosine("causal_spin"),
        "+ spur_out":                     M_cosine("causal_spout"),
        "Ours-full (inductive)":          M_cosine("causal_full"),
        "+ query-time (transductive)":    M_cosine("causal_full", transductive=True),
        "CFSL-φ (frozen random-init)":    M_phi("causal_full", frozen_phi=FROZEN_PHI),
        "CFSL-φ (support-adapted)":       M_phi("causal_full", "classical", adapt=True,
                                               K=TA_SHOT, wd=wd),
    }
    if RUN_QUANTUM:
        methods["QFSL-φ (support-adapted)"] = M_phi("causal_full", "quantum", adapt=True,
                                                    K=TA_SHOT, wd=wd)
    res = {}
    for name, fn in methods.items():
        res[name] = run_method_over_episodes(eps, fn, ckpt_tag=f"headline__{name}")
    _print_table(res, "headline")
    _print_perclass(res)
    _print_deltas(res, "Ours-full (inductive)", adjacent=False)
    return res

def indomain_ablation():
    """[R5][R21] Coverage-controlled ablation on FULLBANK-restricted pool (equal N):
       causal-only vs spur-only (no causal) vs all-patch concat vs causal_full."""
    print("=" * 70, "\n[5d] IN-DOMAIN — coverage-matched ablation (FULLBANK pool)\n", "=" * 70)
    if not bal_full:
        print("  FULLBANK pool unavailable — ablation skipped."); return {}
    eps = {}
    for sd in TA_SEEDS:
        e, _ = gen_episodes(sd, TEST_NOVEL, bal_full, TA_SHOT, patient_of=nih_patient_of)
        eps[sd] = e
    n_ok = sum(len(e) for e in eps.values())
    print(f"  episodes kept (FULLBANK): {n_ok}; all methods see the SAME images [R5].")
    methods = {
        "Causal-only":               M_cosine("causal"),
        "Spur-only (no causal)":     M_cosine("spur_only"),
        "All-patch (concat)":        M_cosine("all"),
        "Causal_full (causal−spur)": M_cosine("causal_full"),
    }
    res = {}
    for name, fn in methods.items():
        res[name] = run_method_over_episodes(eps, fn, ckpt_tag=f"ablation__{name}")
    _print_table(res, "ablation")
    _print_deltas(res, "Causal_full (causal−spur)", adjacent=False)
    return res

def indomain_split_robustness():
    """[#6] Report Ours-full (causal_full, inductive) across several class-split seeds so
       the headline is not conditioned on the single TA_SPLIT_SEED partition. Uses the
       image-disjoint TRAIN split for each class to stay consistent with the main run."""
    print("=" * 70, "\n[5e] IN-DOMAIN — class-split robustness (Ours-full)\n", "=" * 70)
    out = {}
    for ss in range(N_SPLIT_ROBUST):
        rs = np.random.default_rng(TA_SPLIT_SEED + 100 + ss)
        av = [c for c in range(N_CLASSES) if len(pool[c]) >= TA_SHOT + TA_QUERY]
        if len(av) < TA_WAY: continue
        tn = sorted(list(rs.permutation(av))[:TA_WAY])
        trp = {c: _image_split(pool[c], VAL_FRAC, TA_SPLIT_SEED + 50 + c)[0] for c in tn}
        cap = min(len(trp[c]) for c in tn)
        if cap < TA_SHOT + TA_MIN_QUERY: continue
        b = {c: list(rs.permutation(trp[c])[:cap]) for c in tn}
        eps = {}
        for sd in TA_SEEDS:
            eps[sd] = gen_episodes(sd, tn, b, TA_SHOT, patient_of=nih_patient_of)[0]
        r = run_method_over_episodes(eps, M_cosine("causal_full"))
        key = f"split{ss} [" + ",".join(TARGET_CLASSES[c][:4] for c in tn) + "]"
        out[key] = r["auroc_mean"]
        print(f"    {key:<40}{r['auroc_mean']:.4f}")
    vals = [v for v in out.values() if v == v]
    summ = {"per_split": out,
            "mean": float(np.mean(vals)) if vals else float("nan"),
            "std":  float(np.std(vals)) if vals else 0.0, "n_splits": len(vals)}
    print(f"  → Ours-full across {len(vals)} class-splits: "
          f"{summ['mean']:.4f} ± {summ['std']:.4f} (single-split numbers are conditioned "
          f"on TA_SPLIT_SEED={TA_SPLIT_SEED}).")
    return summ

def indomain_ksweep(chosen_wd=None):
    print("=" * 70, "\n[5b] IN-DOMAIN — sample-efficiency sweep K∈" + str(K_SWEEP) + "\n", "=" * 70)
    wd = PHI_WD if chosen_wd is None else chosen_wd
    curves = _dd(dict)
    for K in K_SWEEP:
        eps = {}
        for sd in TA_SEEDS:
            e, _ = gen_episodes(sd, TEST_NOVEL, bal, K, patient_of=nih_patient_of)
            eps[sd] = e
        variants = {
            "Linear probe":        M_linear("causal_full"),   # [R10] anchors the curve
            "ProtoNet":            M_proto_euclid(),
            "Ours-full (metric)":  M_cosine("causal_full"),
            "CFSL-φ (frozen)":     M_phi("causal_full", frozen_phi=FROZEN_PHI),
            "CFSL-φ (adapted)":    M_phi("causal_full", "classical", adapt=True, K=K, wd=wd),
        }
        if RUN_QUANTUM:
            variants["QFSL-φ (adapted)"] = M_phi("causal_full", "quantum", adapt=True, K=K, wd=wd)
        print(f"\n  K={K}:")
        for name, fn in variants.items():
            r = run_method_over_episodes(eps, fn, ckpt_tag=f"ksweep_K{K}__{name}")
            curves[name][K] = r
            print(f"    {name:<22}{r['auroc_mean']:>8.4f}±{r['auroc_std']:.3f}  {_fmt_ci(r)}")
    return curves

def indomain_ttda(chosen_wd=None):
    """[6][R1] TRUE one-change-per-row ladder: each row keeps everything the previous
       row added and changes exactly ONE factor.
         BASELINE        = cosine, inductive
         + transductive  = cosine + query-time refinement
         + φ adapted     = cosine(transductive) + φ eval   (φ is the only new factor)
         + TENT-φ        = cosine(transductive) + φ + TENT (TENT is the only new factor)"""
    print("=" * 70, "\n[5c] IN-DOMAIN — TTDA ladder (control; TRUE one change per row)\n", "=" * 70)
    wd = PHI_WD if chosen_wd is None else chosen_wd
    eps = {}
    for sd in TA_SEEDS:
        e, _ = gen_episodes(sd, TEST_NOVEL, bal, TA_SHOT, patient_of=nih_patient_of)
        eps[sd] = e
    ladder = {
        "BASELINE (inductive metric)": M_cosine("causal_full"),
        "+ transductive (query-time)": M_cosine("causal_full", transductive=True),
        "+ φ support-adapted":         M_phi("causal_full", "classical", adapt=True,
                                             transductive=True, K=TA_SHOT, wd=wd),        # [R1]
        "+ TENT-φ":                    M_phi("causal_full", "classical", adapt=True,
                                             tent=True, transductive=True, K=TA_SHOT, wd=wd),  # [R1]
    }
    res = {}
    for name, fn in ladder.items():
        res[name] = run_method_over_episodes(eps, fn, ckpt_tag=f"ttda__{name}")
        r = res[name]
        print(f"  {name:<30}{r['auroc_mean']:>8.4f}±{r['auroc_std']:.3f}  {_fmt_ci(r)}")
    _print_deltas(res, None, adjacent=True)             # [R6] adjacent-row significance
    return res

## §6 — CheXpert encode (frozen CheXzero)
Frontal-only, U-Ignore handling, OOM-safe batched flush, and key-load diagnostics. Cached to `chexpert_frontal_cz512.pt`.

In [ ]:
# ==============================================================================
# SECTION 6 — CHEXPERT ENCODE (GPU) + single-label pools
# ==============================================================================
def load_chexzero():
    m, prep = clip.load("ViT-B/32", device=Device, jit=False)
    st = torch.load(CHEXZERO_CKPT, map_location=Device, weights_only=False)
    if isinstance(st, dict):
        for k in ("state_dict", "model", "model_state_dict", "net"):
            if k in st: st = st[k]; break
    clean = {k.replace("module.", "").replace("model.", ""): v for k, v in st.items()}
    ret = m.load_state_dict(clean, strict=False)        # [26] log missing/unexpected
    miss = list(getattr(ret, "missing_keys", [])); unex = list(getattr(ret, "unexpected_keys", []))
    print(f"  [26] CheXzero load: {len(miss)} missing, {len(unex)} unexpected keys.")
    if len(miss) > 20:
        print(f"       ⚠️ {len(miss)} missing keys — checkpoint/key-cleaning may be off; "
              f"e.g. {miss[:5]}")
    m.eval().to(Device)
    for p in m.parameters(): p.requires_grad_(False)
    return m, prep

def encode_chexpert():
    print("=" * 70, "\n[6] Encoding CheXpert frontal via frozen CheXzero\n", "=" * 70)
    if os.path.exists(CX_CACHE):
        print("  loading CheXpert cache"); return torch.load(CX_CACHE, weights_only=False)
    csv = (glob.glob(f"{CHEXPERT_DIR}/**/train.csv", recursive=True) or [None])[0]
    assert csv, f"❌ CheXpert train.csv not found under {CHEXPERT_DIR}"
    root = os.path.dirname(os.path.dirname(csv)); df = pd.read_csv(csv)
    pcol = "Path" if "Path" in df.columns else df.columns[0]
    def cx_col(c):
        for a in CX_ALIASES.get(c, [c]):
            if a in df.columns: return a
        return None
    cols = [cx_col(c) for c in TARGET_CLASSES]
    def resolve(rel):
        for cand in (os.path.join(root, rel), os.path.join(CHEXPERT_DIR, rel)):
            if os.path.isfile(cand): return cand
        return None
    def pat(rel):                                        # [25] re imported at module scope
        m = re.search(r"(patient\d+)", str(rel)); return m.group(1) if m else _bn(rel)
    rows = []
    for _, r in df.iterrows():
        rel = str(r[pcol])
        if _bn(rel).startswith("._") or "frontal" not in rel.lower(): continue
        ap = resolve(rel)
        if ap is None: continue
        y = np.full(N_CLASSES, np.nan, np.float32)
        for i, col in enumerate(cols):
            if col is not None and not pd.isna(r[col]):
                v = float(r[col]); y[i] = np.nan if v == -1.0 else v   # U-Ignore
        rows.append((ap, pat(rel), y))
        if CX_SUBSET_ENCODE and len(rows) >= CX_SUBSET_ENCODE: break
    print(f"  frontal rows to encode: {len(rows):,}")
    assert rows, "❌ 0 CheXpert frontal rows resolved."
    cz, prep = load_chexzero()
    F512 = torch.zeros(len(rows), 512); buf, idx = [], []
    @torch.no_grad()
    def flush():                                        # [18] OOM-safe: always clear
        if not buf: return
        try:
            ts = torch.stack([prep(im) for im in buf]).to(Device)
            with torch.amp.autocast("cuda", enabled=torch.cuda.is_available()):
                f = F.normalize(cz.encode_image(ts).float(), dim=-1).cpu()
            for j, ix in enumerate(idx): F512[ix] = f[j]
        finally:
            buf.clear(); idx.clear()
    for i, (ap, _, _) in enumerate(tqdm(rows, desc="  encode")):
        try:
            buf.append(Image.open(ap).convert("RGB")); idx.append(i)
            if len(buf) >= 64: flush()
        except Exception: continue
    flush()
    pack = {"feats": F512, "labels": np.stack([r[2] for r in rows]),
            "patients": [r[1] for r in rows]}
    torch.save(pack, CX_CACHE); print(f"  ✅ cached {tuple(F512.shape)}")
    return pack

## §7 — Cross-domain runner (NIH→CheXpert)
Build-up ladder, TTDA ladder, and the cross-domain K-sweep. Anchoring is aligned to true `ways`; classes restricted to CheXpert∧NIH-anchor.

In [ ]:
# ==============================================================================
# SECTION 7 — CROSS-DOMAIN RUNNER (NIH -> CheXpert)
# ==============================================================================
CROSS_CTX = {}            # [R20] stashes cxfeat/anchor_stack/episodes for the dump cell

def _clean_feats(x):
    """spur_out cleaning applied to raw features so the φ path sits on the SAME
       cleaned base as the metric ladder (keeps the ladder one-change-per-row).
       NOTE (#8): cross-domain cleaning subtracts the DATASET-LEVEL mean `spout_dir`
       (CheXpert has no per-image spur bank) — NOT the per-image subtraction used by the
       in-domain `causal_full` variant. Documented as distinct in the header."""
    if spout_dir is None: return x
    return F.normalize(x - TA_BETA * spout_dir.to(x.device), dim=-1)

def run_crossdomain():
    pack = encode_chexpert()
    CXF, CXY, CXP = pack["feats"], pack["labels"], pack["patients"]
    diagnostics = {"cx_load_failures": int(pack.get("n_load_failures", 0)),   # [#10]
                   "cx_rows": int(pack.get("n_rows", len(CXP)))}
    cx_pool = _dd(list); cx_pat = {}
    for i in range(len(CXP)):
        pos = [c for c in range(N_CLASSES) if CXY[i, c] == 1.0]
        if len(pos) == 1:
            cx_pool[pos[0]].append(i); cx_pat[i] = CXP[i]
    avail = [c for c in range(N_CLASSES) if len(cx_pool[c]) >= (TA_SHOT + TA_QUERY) * 2]
    print("  CheXpert single-label pool: " +
          "  ".join(f"{TARGET_CLASSES[c][:4]}={len(cx_pool[c])}" for c in range(N_CLASSES)))

    # NIH source causal prototypes (512-space) for anchoring
    nih_proto = {c: F.normalize(torch.stack([sig_causal[k] for k in pool[c]]).mean(0), dim=0)
                 for c in range(N_CLASSES) if pool[c]}
    avail = [c for c in avail if c in nih_proto]        # [2] need CheXpert data ∧ NIH anchor
    print(f"  usable cross-domain classes (CheXpert ∧ NIH-anchor): "
          f"{[TARGET_CLASSES[c] for c in avail]}")
    if len(avail) < TA_WAY:
        print(f"  ⚠️ <{TA_WAY} usable CheXpert classes — cross-domain skipped."); return None

    # [#3] IMAGE-disjoint (patient-aware) train/val split of the CheXpert pool, BEFORE
    #      balancing, so the α sweep tunes on images absent from every reported episode.
    def _cx_split(items, frac, seed):
        rp = np.random.default_rng(seed); it = list(rp.permutation(items))
        nval = int(round(len(it) * frac)); val = it[:nval]
        vpat = {cx_pat[i] for i in val}
        return [i for i in it[nval:] if cx_pat[i] not in vpat], val
    cx_train, cx_val = {}, {}
    for c in avail:
        tr, va = _cx_split(cx_pool[c], VAL_FRAC, TA_SPLIT_SEED + 60 + c)
        cx_train[c], cx_val[c] = tr, va
    cap = min(len(cx_train[c]) for c in avail)
    _dr = np.random.default_rng(TA_SPLIT_SEED + 2)
    cx_bal = {c: list(_dr.permutation(cx_train[c])[:cap]) for c in avail}
    _capv = min(len(cx_val[c]) for c in avail)
    cx_val_bal = {}
    if _capv >= TA_SHOT + TA_MIN_QUERY:
        _drv = np.random.default_rng(TA_SPLIT_SEED + 4)
        cx_val_bal = {c: list(_drv.permutation(cx_val[c])[:_capv]) for c in avail}
    print(f"  balanced CheXpert TRAIN pool to {cap}/class; image-disjoint val "
          f"{_capv}/class [#3]")

    def cxfeat(idx_list): return torch.stack([CXF[i] for i in idx_list]).to(Device)
    def anchor_stack(ways):                             # [1][2] driven by true class ids
        assert all(c in nih_proto for c in ways), f"missing NIH anchor for {ways}"
        a = torch.stack([nih_proto[c] for c in ways]).to(Device)
        assert a.shape == (len(ways), 512), f"anchor shape {tuple(a.shape)} != ({len(ways)},512)"  # [R18]
        return a

    # ---- episodes per K (shared by every method + the dump cell) [R20] ------
    cx_eps = {}
    for K in K_SWEEP:
        eK, dK = {}, 0
        for sd in TA_SEEDS:
            e, d = gen_episodes(sd, avail, cx_bal, K, patient_of=cx_pat); eK[sd] = e; dK += d
        cx_eps[K] = eK
        if K == TA_SHOT:
            OUT_OVERLAP["crossdomain_K5"] = episode_overlap_fraction(eK)   # [R8]
    eps5 = cx_eps.get(TA_SHOT) or cx_eps[K_SWEEP[-1]]

    # ---- method closures — signature (sk, sy, qk, qy, ways, rng, ep_seed) ----
    def CM_baseline(sk, sy, qk, qy, ways, rng, ep_seed):
        return proto_cosine(cxfeat(sk), sy.to(Device), cxfeat(qk), qy, len(ways))
    def CM_clean(sk, sy, qk, qy, ways, rng, ep_seed):
        return proto_cosine(cxfeat(sk), sy.to(Device), cxfeat(qk), qy, len(ways), clean=True)
    def CM_linear(sk, sy, qk, qy, ways, rng, ep_seed):  # [R10]
        return linear_probe(cxfeat(sk), sy.to(Device), cxfeat(qk), qy, len(ways))
    def CM_anchor_a(alpha):                             # α threaded, no global mutation
        def fn(sk, sy, qk, qy, ways, rng, ep_seed):
            return proto_cosine(cxfeat(sk), sy.to(Device), cxfeat(qk), qy, len(ways),
                                clean=True, anchor=anchor_stack(ways), alpha=alpha)
        return fn
    def CM_qt_a(alpha):
        def fn(sk, sy, qk, qy, ways, rng, ep_seed):
            return proto_cosine(cxfeat(sk), sy.to(Device), cxfeat(qk), qy, len(ways),
                                clean=True, anchor=anchor_stack(ways),
                                transductive=True, alpha=alpha)
        return fn
    def CM_phi(tent=False, transductive=False, K=TA_SHOT, alpha=None):
        def fn(sk, sy, qk, qy, ways, rng, ep_seed):
            sx = _clean_feats(cxfeat(sk)); qx = _clean_feats(cxfeat(qk))
            anc = _clean_feats(anchor_stack(ways)); syd = sy.to(Device)
            phi = adapt_phi(build_phi("classical", init_seed=ep_seed), sx, syd,
                            len(ways), K, rng=rng)                      # [R2][#2]
            if tent: phi = tent_phi(phi, sx, syd, qx, len(ways))
            r = eval_phi(phi, sx, syd, qx, qy, len(ways),
                         anchor=anc, transductive=transductive, alpha=alpha)   # [R1][R4]
            del phi; return r
        return fn

    CROSS_CTX.update(cxfeat=cxfeat, anchor_stack=anchor_stack, eps=cx_eps, avail=avail)

    # ---- [R11] α sweep on the IMAGE-DISJOINT validation pool [#3] ------------
    print("=" * 70, "\n[R11] CROSS-DOMAIN — CX_ANCHOR_ALPHA sweep (image-held-out)\n", "=" * 70)
    alpha_table, best_a = {}, None
    if cx_val_bal:
        val_eps = {8891: gen_episodes(8891, avail, cx_val_bal, TA_SHOT, patient_of=cx_pat)[0]}
        for a in ALPHA_SWEEP:
            r = run_method_over_episodes(val_eps, CM_qt_a(a), ckpt_tag=f"cx_alpha{a}")
            alpha_table[str(a)] = r["auroc_mean"]
            print(f"    α={a:<5} val-AUROC={r['auroc_mean']:.4f}")
            if best_a is None or (r["auroc_mean"] == r["auroc_mean"] and r["auroc_mean"] > best_a[1]):
                best_a = (a, r["auroc_mean"])
        chosen_alpha = best_a[0]
        print(f"  → chosen CX_ANCHOR_ALPHA = {chosen_alpha} (val AUROC={best_a[1]:.4f}); "
              f"threaded explicitly (no global mutation). Image-disjoint holdout [#3].")
    else:
        chosen_alpha = CX_ANCHOR_ALPHA
        print(f"  no image-disjoint val pool — using default α={chosen_alpha}.")

    # concrete closures for the reported tables (α locked & threaded)
    CM_anchor = CM_anchor_a(chosen_alpha); CM_qt = CM_qt_a(chosen_alpha)

    def _run5(fn, tag): return run_method_over_episodes(eps5, fn, ckpt_tag=tag)

    # ---- build-up ladder (K=5) ----------------------------------------------
    print("=" * 70, "\n[7] CROSS-DOMAIN NIH→CheXpert — build-up ladder (K=5)\n", "=" * 70)
    build = {"Linear probe (CheXpert)":     CM_linear,
             "BASELINE (CheXpert proto)":    CM_baseline,
             "+ spur_out clean (global)":    CM_clean,
             "+ NIH-anchor (ours)":          CM_anchor,
             "+ query-time (transductive)":  CM_qt,
             "CFSL-φ (adapted, full stack)": CM_phi(transductive=True, alpha=chosen_alpha)}
    cd = {}
    for name, fn in build.items():
        cd[name] = _run5(fn, f"cx_build__{name}")
        r = cd[name]
        print(f"  {name:<32}{r['auroc_mean']:>8.4f}±{r['auroc_std']:.3f}  {_fmt_ci(r)}")
    _print_deltas(cd, "+ query-time (transductive)", adjacent=False)

    # ---- [6][R1] TTDA ladder: TRUE one change per row -----------------------
    print("=" * 70, "\n[7b] CROSS-DOMAIN — TTDA ladder (TRUE one change per row)\n", "=" * 70)
    ladder = {"BASE (clean+anchor, inductive)": CM_anchor,
              "+ transductive (query-time)":    CM_qt,
              "+ φ support-adapted":            CM_phi(transductive=True, alpha=chosen_alpha),   # [R1]
              "+ TENT-φ":                       CM_phi(tent=True, transductive=True,
                                                       alpha=chosen_alpha)}                       # [R1]
    cd_ttda = {}
    for name, fn in ladder.items():
        cd_ttda[name] = _run5(fn, f"cx_ttda__{name}")
        r = cd_ttda[name]
        print(f"  {name:<32}{r['auroc_mean']:>8.4f}±{r['auroc_std']:.3f}  {_fmt_ci(r)}")
    _print_deltas(cd_ttda, None, adjacent=True)         # [R6]

    # ---- [27] cross-domain K-sweep (honestly labelled) ----------------------
    print("=" * 70, "\n[7c] CROSS-DOMAIN — sample-efficiency sweep K∈" + str(K_SWEEP) + "\n", "=" * 70)
    cd_ksweep = _dd(dict)
    for K in K_SWEEP:
        epsK = cx_eps[K]
        variants = {"CheXpert cosine (metric)": CM_baseline,       # true zero-param metric
                    "Full pipeline (no φ)":     CM_qt,             # NOT metric-only — relabelled
                    "Linear probe":             CM_linear,         # [R10]
                    "CFSL-φ (adapted)":         CM_phi(transductive=True, K=K, alpha=chosen_alpha)}
        print(f"\n  K={K}:")
        for name, fn in variants.items():
            r = run_method_over_episodes(epsK, fn, ckpt_tag=f"cx_ksweep_K{K}__{name}")
            cd_ksweep[name][K] = r
            print(f"    {name:<26}{r['auroc_mean']:>8.4f}±{r['auroc_std']:.3f}  {_fmt_ci(r)}")

    def _strip(d):    # drop underscore-prefixed internals before returning for JSON
        return {k: v for k, v in d.items() if not str(k).startswith("_")}
    return {"buildup":  {n: _strip(r) for n, r in cd.items()},
            "ttda":     {n: _strip(r) for n, r in cd_ttda.items()},
            "ksweep":   {k: {str(kk): _strip(vv) for kk, vv in v.items()}
                         for k, v in cd_ksweep.items()},
            "alpha_sweep": {"sweep": alpha_table, "chosen_alpha": chosen_alpha,
                            "holdout": "image-disjoint validation pool [#3] (val seed 8891)"},
            "diagnostics": diagnostics}

## §8 — Episode dump for the quantum notebook
Saves the **exact** in-domain episodes as CPU feature tensors (`episodes/indomain_K*.pt`) so `φ_q` scores identical episodes offline.

In [ ]:
# ==============================================================================
# SECTION 8 — EPISODE DUMP FOR THE QUANTUM NOTEBOOK  (required outputs)
# ==============================================================================
def _dump_episode_tensors(name, episodes_by_seed, feat_fn, anchor_fn=None):
    """Save exact episodes as CPU feature tensors so φ_q scores identical episodes.
       feat_fn(keys)->Tensor; anchor_fn(ways)->Tensor or None."""
    payload = []
    for sd, eps in episodes_by_seed.items():
        for (sk, sy, qk, qy, ways) in eps:
            item = {"seed": int(sd), "ways": [int(w) for w in ways],
                    "sx": feat_fn(sk).cpu(), "sy": torch.tensor(sy),
                    "qx": feat_fn(qk).cpu(), "qy": torch.tensor(qy)}
            if anchor_fn is not None: item["anchor"] = anchor_fn(ways).cpu()
            payload.append(item)
    path = f"{EPISODE_DIR}/{name}.pt"
    torch.save(payload, path)
    print(f"  [dump] {name}: {len(payload)} episodes -> {path}")

def save_indomain_episodes():
    if not (SAVE_EPISODES and INDOMAIN_OK): return
    print("=" * 70, "\n[8] Dumping in-domain episodes (φ_q reload)\n", "=" * 70)
    for K in K_SWEEP:
        eps = {}
        for sd in TA_SEEDS:
            e, _ = gen_episodes(sd, TEST_NOVEL, bal, K, patient_of=nih_patient_of)
            eps[sd] = e
        _dump_episode_tensors(f"indomain_K{K}", eps,
                              lambda keys: stack_feats(keys, "causal_full"))

def save_crossdomain_episodes():
    """[R20] Dump cross-domain episodes in the SAME tensor format, INCLUDING the per-
       episode NIH `anchor`, so the quantum notebook can replay cross-domain CFSL vs
       QFSL on identical support/query splits. Requires run_crossdomain() to have run."""
    if not (SAVE_EPISODES and CROSS_CTX.get("eps")):
        print("  [8b] cross-domain episodes not dumped (cross-domain unavailable).")
        return
    print("=" * 70, "\n[8b] Dumping cross-domain episodes (+anchor) for φ_q\n", "=" * 70)
    cxfeat = CROSS_CTX["cxfeat"]; anchor_stack = CROSS_CTX["anchor_stack"]
    for K, eps in CROSS_CTX["eps"].items():
        _dump_episode_tensors(f"crossdomain_K{K}", eps, cxfeat, anchor_fn=anchor_stack)

## §9 — Run & save all matrices
Drives everything and writes `fsl_cfsl_qfsl_results.json` (NaN→null) + all CSVs.

In [ ]:
# ==============================================================================
# SECTION 9 — RUN + SAVE ALL MATRICES
# ==============================================================================
def _san(o):                                            # [24] NaN/Inf -> None; strip _internal
    if isinstance(o, dict):
        return {k: _san(v) for k, v in o.items() if not str(k).startswith("_")}
    if isinstance(o, list):  return [_san(v) for v in o]
    if isinstance(o, float): return None if (math.isnan(o) or math.isinf(o)) else o
    if isinstance(o, (np.floating,)): return _san(float(o))
    if isinstance(o, (np.integer,)):  return int(o)
    return o

# [R15] reproducibility block — libraries, CUDA, GPU, RNG state
def _repro_block():
    import PIL
    try: cz_ver = clip.__version__
    except Exception: cz_ver = "openai/CLIP@" + CLIP_COMMIT
    return {
        "python": sys.version.split()[0], "platform": platform.platform(),
        "torch": torch.__version__, "numpy": np.__version__, "pandas": pd.__version__,
        "sklearn": sklearn.__version__, "scipy": scipy.__version__, "pillow": PIL.__version__,
        "clip": cz_ver, "clip_commit": CLIP_COMMIT,
        "cuda": torch.version.cuda if torch.cuda.is_available() else None,
        "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
        "torch_rng_state_sha": hash(tuple(torch.get_rng_state().tolist()[:64])),
        "np_rng_seed": SEED, "bootstrap_n": BOOTSTRAP_N, "boot_seed": BOOT_SEED,
    }

OUT = {"config": {"n_qubits": N_QUBITS, "phi_depth": PHI_DEPTH, "way": TA_WAY,
                   "shot": TA_SHOT, "query": TA_QUERY, "min_query": TA_MIN_QUERY,
                   "episodes": TA_EPISODES, "seeds": TA_SEEDS, "k_sweep": K_SWEEP,
                   "cos_temp": COS_TEMP, "tent_temp": TENT_TEMP, "qt_margin": TA_QT_MARGIN,
                   "phi_lr": PHI_LR, "phi_wd_default": PHI_WD, "dry_run": DRY_RUN,
                   "test_novel": [TARGET_CLASSES[c] for c in TEST_NOVEL],
                   "reported_score": "raw cosine similarity (AUROC); argmax (F1)",
                   "ci_method": "cluster bootstrap percentile — resample seeds then "
                                "episodes (accounts for [R8] cross-seed overlap) [#5]",
                   "significance": "paired Wilcoxon signed-rank on per-episode AUROC, "
                                   "Holm-Bonferroni corrected per family [#4]",
                   "val_split": f"image-disjoint, VAL_FRAC={VAL_FRAC} held out before "
                                f"balancing [#3]"},
       "contamination": CONTAMINATION,          # [R14]
       "meta_training": META_TRAINING,          # [R13]
       "reproducibility": _repro_block()}       # [R15]

def _save_csv(path, header, rows):
    with open(path, "w", newline="") as f:
        w = _csv.writer(f); w.writerow(header); [w.writerow(r) for r in rows]

def _cell(d, k):                                        # [new-issue guard] no KeyError
    r = d.get(k)
    if r is None or r.get("auroc_mean") is None or np.isnan(r.get("auroc_mean", float("nan"))):
        return "nan"
    return f"{r['auroc_mean']:.4f}"

# ---- run everything ----------------------------------------------------------
_sanity_check()                                         # [R18] hard asserts first

if RUN_INDOMAIN and INDOMAIN_OK:
    wd_info = indomain_wd_sweep()                       # [R12] chosen WD is THREADED, not global
    OUT["wd_sweep"] = wd_info
    _wd = wd_info["chosen_wd"]
    head = indomain_headline(chosen_wd=_wd); OUT["indomain_headline"] = head
    abl = indomain_ablation();  OUT["indomain_ablation"] = abl        # [R5][R21]
    split_rob = indomain_split_robustness(); OUT["indomain_split_robustness"] = split_rob  # [#6]
    curves = indomain_ksweep(chosen_wd=_wd)
    OUT["indomain_ksweep"] = {k: {str(kk): vv for kk, vv in v.items()} for k, v in curves.items()}
    ttda = indomain_ttda(chosen_wd=_wd); OUT["indomain_ttda"] = ttda
    save_indomain_episodes()
    OUT["seed_overlap"] = OUT_OVERLAP                   # [R8]
    full = head["Ours-full (inductive)"]["auroc_mean"]
    _save_csv(f"{OUT_DIR}/indomain_headline.csv",
              ["method", "auroc_mean", "auroc_std", "boot_lo", "boot_hi",
               "f1_mean", "f1_std", "n_episodes", "wall_sec", "delta_vs_full"],
              [[n, f"{r['auroc_mean']:.4f}", f"{r['auroc_std']:.4f}",
                f"{r.get('auroc_boot_ci95_lo', float('nan')):.4f}",
                f"{r.get('auroc_boot_ci95_hi', float('nan')):.4f}",
                f"{r['f1_mean']:.4f}", f"{r['f1_std']:.4f}", r.get('n_episodes', 0),
                f"{r.get('wall_sec', 0):.1f}", f"{r['auroc_mean']-full:+.4f}"]
               for n, r in head.items()])
    if abl:
        _save_csv(f"{OUT_DIR}/indomain_ablation.csv",
                  ["variant", "auroc_mean", "auroc_std", "n_episodes"],
                  [[n, f"{r['auroc_mean']:.4f}", f"{r['auroc_std']:.4f}", r.get('n_episodes', 0)]
                   for n, r in abl.items()])
    _save_csv(f"{OUT_DIR}/indomain_ksweep.csv",
              ["method"] + [f"K{k}" for k in K_SWEEP],
              [[n] + [_cell(curves[n], k) for k in K_SWEEP] for n in curves])
    _save_csv(f"{OUT_DIR}/indomain_ttda.csv",
              ["step", "auroc_mean", "auroc_std"],
              [[n, f"{r['auroc_mean']:.4f}", f"{r['auroc_std']:.4f}"] for n, r in ttda.items()])

if RUN_CROSSDOMAIN:
    cd = run_crossdomain()
    if cd:
        OUT["crossdomain"] = cd
        save_crossdomain_episodes()                     # [R20]
        OUT["seed_overlap"] = OUT_OVERLAP
        _save_csv(f"{OUT_DIR}/crossdomain_buildup.csv",
                  ["config", "auroc_mean", "auroc_std"],
                  [[n, f"{r['auroc_mean']:.4f}", f"{r['auroc_std']:.4f}"] for n, r in cd["buildup"].items()])
        _save_csv(f"{OUT_DIR}/crossdomain_ttda.csv",
                  ["step", "auroc_mean", "auroc_std"],
                  [[n, f"{r['auroc_mean']:.4f}", f"{r['auroc_std']:.4f}"] for n, r in cd["ttda"].items()])
        _save_csv(f"{OUT_DIR}/crossdomain_ksweep.csv",
                  ["method"] + [f"K{k}" for k in K_SWEEP],
                  [[n] + [(cd["ksweep"][n].get(str(k), {}).get("auroc_mean") is not None
                          and f"{cd['ksweep'][n][str(k)]['auroc_mean']:.4f}" or "nan")
                          for k in K_SWEEP]
                   for n in cd["ksweep"]])

with open(f"{OUT_DIR}/fsl_cfsl_qfsl_results.json", "w") as f:
    json.dump(_san(OUT), f, indent=2)                   # [24] spec-valid JSON

print("\n" + "=" * 70)
print("✅ DONE — artifacts in /kaggle/working:")
print("   fsl_cfsl_qfsl_results.json  (config, contamination[R14], meta_training[R13],")
print("                                seed_overlap[R8], reproducibility[R15], all tables)")
print("   indomain_headline.csv | indomain_ablation.csv | indomain_ksweep.csv | indomain_ttda.csv")
print("   crossdomain_buildup.csv | crossdomain_ttda.csv | crossdomain_ksweep.csv")
print("   chexpert_frontal_cz512.pt | checkpoints/*.pt  (per-seed resume [R16])")
print("   episodes/indomain_K*.pt  +  episodes/crossdomain_K*.pt  ← identical episodes for φ_q")
print("=" * 70)